# Rezultati za članek:
*Impact of Mechanical Stress on Electrochemical Impedance Spectra and Incremental Capacity of the Lithium-Ion Battery*

Poženi prvo file ```install_pybamm.ipynb```, da si naložiš modificirano verzijo PyBaMM-a.

In [ ]:
# aktiviraj myenv okolje
import pybamm
import numpy as np
import matplotlib.pyplot as plt
import time
from ipywidgets import interact, FloatSlider
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from scipy import interpolate
from scipy import optimize
from joblib import Parallel, delayed
from scipy.signal import savgol_filter

In [ ]:
# Shrani slike
save_fig = False 

# Shrani animacijo
save_ani = False

# Formatiranje grafov
# Nastavitev velikosti grafa (širina, višina v inch-ih)
fig_size = figsize=(8, 6)  

# Nastavitev velikosti pisave
plt.rcParams.update({
    'font.size': 24,           # Osnovna velikost pisave
    'axes.titlesize': 24,      # Velikost naslova grafa
    'axes.labelsize': 24,      # Velikost pisave na osi
    'xtick.labelsize': 21,     # Velikost številk na x osi
    'ytick.labelsize': 21,     # Velikost številk na y osi
    'legend.fontsize': 22,     # Velikost pisave v legendi
})

Load results

In [ ]:
sol_DFN = pybamm.load("NMC_LiC6 results/NMC_LiC6_0Pa_discharge.pkl")
sol_DFN_1 = pybamm.load("NMC_LiC6 results/NMC_LiC6_1MPa_discharge.pkl")
sol_DFN_2 = pybamm.load("NMC_LiC6 results/NMC_LiC6_10MPa_discharge.pkl")
sol_DFN_C = pybamm.load("NMC_LiC6 results/NMC_LiC6_0Pa_charge.pkl")
sol_DFN_1_C = pybamm.load("NMC_LiC6 results/NMC_LiC6_1MPa_charge.pkl")
sol_DFN_2_C = pybamm.load("NMC_LiC6 results/NMC_LiC6_10MPa_charge.pkl")

## OCV

In [ ]:
model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                        "particle": "Fickian diffusion",
                                                },
                                        )
param0 = pybamm.ParameterValues("Mohtat2020_Mech") 
param0.update(
    {"Hydrostatic stress [Pa]": 0,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-4,
     },
    check_already_exists=False ,
    
)


param1 = pybamm.ParameterValues("Mohtat2020_Mech") 
param1.update(
    {"Hydrostatic stress [Pa]": 1e6,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-4},
    check_already_exists=False  
)


param2 = pybamm.ParameterValues("Mohtat2020_Mech") 
param2.update(
    {"Hydrostatic stress [Pa]": 1e7,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-4},
    check_already_exists=False  
)


var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

# experiment = pybamm.Experiment(["Discharge at C/50 until 2.75 V"])
# experiment_C = pybamm.Experiment(["Charge at C/50 until 4.2 V"])

experiment = pybamm.Experiment([pybamm.step.string(
    "Discharge at C/50 until 2.75 V", period="1 s"
)
])
experiment_C = pybamm.Experiment([pybamm.step.string(
    "Charge at C/50 until 4.2 V", period="1 s"
)
])

solver = pybamm.IDAKLUSolver()  


# # discharge battery
# sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param0, var_pts=var_pts, solver=solver, experiment=experiment 
#                               )
# sim_DFN_1 = pybamm.Simulation(model= model, parameter_values=param1, var_pts=var_pts, solver=solver, experiment=experiment 
#                               )
# sim_DFN_2 = pybamm.Simulation(model= model, parameter_values=param2, var_pts=var_pts, solver=solver, experiment=experiment 
#                               )

# # charge battery
# sim_DFN_0_C = pybamm.Simulation(model= model, parameter_values=param0, var_pts=var_pts, experiment=experiment_C, solver=solver)
# sim_DFN_1_C = pybamm.Simulation(model= model, parameter_values=param1, var_pts=var_pts, experiment=experiment_C, solver=solver)
# sim_DFN_2_C = pybamm.Simulation(model= model, parameter_values=param2, var_pts=var_pts, experiment=experiment_C, solver=solver)

######################
t_eval = [0, 3600]
# t_eval = np.linspace(0, 0.3, 1000)  
######################

In [ ]:
# start = time.time()
# sol_DFN = sim_DFN_0.solve(initial_soc=1)
# end = time.time()
# print(f"Simulacija DFN_0 zaključena v {end - start:.2f} s.")

# start = time.time()
# sol_DFN_1 = sim_DFN_1.solve(initial_soc=1)
# end = time.time()
# print(f"Simulacija DFN_1 zaključena v {end - start:.2f} s.")

# start = time.time()
# sol_DFN_2 = sim_DFN_2.solve(initial_soc=1)
# end = time.time()
# print(f"Simulacija DFN_2 zaključena v {end - start:.2f} s.")

# start = time.time()
# sol_DFN_C = sim_DFN_0_C.solve(initial_soc=0)  
# end = time.time()
# print(f"Simulacija DFN_0 zaključena v {end - start:.2f} s.")

# start = time.time()
# sol_DFN_1_C = sim_DFN_1_C.solve(initial_soc=0)
# end = time.time()
# print(f"Simulacija DFN_1 zaključena v {end - start:.2f} s.")

# start = time.time()
# sol_DFN_2_C = sim_DFN_2_C.solve(initial_soc=0)
# end = time.time()
# print(f"Simulacija DFN_2 zaključena v {end - start:.2f} s.")


# sol_DFN.save("NMC_LiC6 results/NMC_LiC6_0Pa_discharge.pkl")
# sol_DFN_1.save("NMC_LiC6 results/NMC_LiC6_1MPa_discharge.pkl")
# sol_DFN_2.save("NMC_LiC6 results/NMC_LiC6_10MPa_discharge.pkl")
# sol_DFN_C.save("NMC_LiC6 results/NMC_LiC6_0Pa_charge.pkl")
# sol_DFN_1_C.save("NMC_LiC6 results/NMC_LiC6_1MPa_charge.pkl")
# sol_DFN_2_C.save("NMC_LiC6 results/NMC_LiC6_10MPa_charge.pkl")

In [ ]:
Q_discharged = sol_DFN["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD = (Q_discharged / Q_max)
SOC = 1 - DOD

Q_discharged = sol_DFN_1["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD_1 = (Q_discharged / Q_max)
SOC_1 = 1 - DOD_1

Q_discharged = sol_DFN_2["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD_2 = (Q_discharged / Q_max)
SOC_2 = 1 - DOD_2

Q_discharged = sol_DFN_C["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD_C = (Q_discharged / Q_max)

Q_discharged = sol_DFN_1_C["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD_1_C = (Q_discharged / Q_max)

Q_discharged = sol_DFN_2_C["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD_2_C = (Q_discharged / Q_max)

# x_val = [DOD, 
#          DOD_1,
#         DOD_2
#          ]

x_val = [SOC,
         SOC_1,
        SOC_2
         ]

y_val = [sol_DFN["Battery voltage [V]"].entries, 
         sol_DFN_1["Battery voltage [V]"].entries,
        sol_DFN_2["Battery voltage [V]"].entries,
         ]

# y_val = [sol_DFN["Battery open-circuit voltage [V]"].entries, 
#          sol_DFN_1["Battery open-circuit voltage [V]"].entries,
#         sol_DFN_2["Battery open-circuit voltage [V]"].entries
#          ]


plt.figure(figsize=fig_size)

for i in range(len(x_val)):
    plt.plot(x_val[i], y_val[i], label=[fr"$\sigma_{{\mathrm{{h}}}}$ = 0 Pa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = 1 MPa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = 10 MPa", 
                                        ][i],
                                linestyle=['-', '--', '-.' ][i]

                                        )
    
# # draw line at soc = 0.5
# plt.axvline(x=0.5, color='gray', linestyle='--', label='SOC = 0.5')

# # draw line at 3.7156 V
# plt.axhline(y=3.7156, color='gray', linestyle='--', label='V = 3.7156 V')

plt.xlabel("SOC [/]")
plt.ylabel("V [V]")
# plt.legend(loc="upper right", bbox_to_anchor=(1.5, 0.9))
plt.legend()
# plt.grid()

# save to eps
# plt.savefig("NMC_LiC6_OCV.eps", format='eps',dpi= 300, bbox_inches='tight')

Discharge

In [ ]:
Q_discharged_0Pa = sol_DFN["Discharge capacity [A.h]"].entries
Q_discharged_1MPa = sol_DFN_1["Discharge capacity [A.h]"].entries
Q_discharged_10MPa = sol_DFN_2["Discharge capacity [A.h]"].entries

Q_discharged_0Pa_C = sol_DFN_C["Discharge capacity [A.h]"].entries
Q_discharged_1MPa_C = sol_DFN_1_C["Discharge capacity [A.h]"].entries
Q_discharged_10MPa_C = sol_DFN_2_C["Discharge capacity [A.h]"].entries

x_val = [
        Q_discharged_0Pa,
        Q_discharged_1MPa,
        Q_discharged_10MPa,
        # Q_discharged_0Pa_C*(-1),
        # Q_discharged_1MPa_C*(-1),
        # Q_discharged_10MPa_C*(-1),        
         ]

y_val = [
        sol_DFN["Battery voltage [V]"].entries, 
        sol_DFN_1["Battery voltage [V]"].entries,
        sol_DFN_2["Battery voltage [V]"].entries,
        # sol_DFN_C["Battery voltage [V]"].entries, 
        # sol_DFN_1_C["Battery voltage [V]"].entries,
        # sol_DFN_2_C["Battery voltage [V]"].entries,
         ]


plt.figure(figsize=fig_size)

for i in range(len(x_val)):
    plt.plot(x_val[i], y_val[i], label=[fr"$\sigma_{{\mathrm{{h}}}}$ = 0 Pa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = 1 MPa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = 10 MPa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = 0 Pa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = 1 MPa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = 10 MPa",
                                        ][i],
                                linestyle=['-', '--', '-', '-', '--', '-' ][i]

                                        )
    
# # draw line at soc = 0.5
# plt.axvline(x=0.5, color='gray', linestyle='--', label='SOC = 0.5')

# # draw line at 3.7156 V
# plt.axhline(y=3.7156, color='gray', linestyle='--', label='V = 3.7156 V')

plt.xlabel("Q [Ah]")
plt.ylabel("V [V]")
# plt.legend(loc="upper right", bbox_to_anchor=(1.5, 0.9))
plt.legend()
# plt.grid()

# save to eps
# plt.savefig("NMC_LiC6_VQ_charge.eps", format='eps',dpi= 300, bbox_inches='tight')

Charge

In [ ]:
Q_discharged_0Pa = sol_DFN["Discharge capacity [A.h]"].entries
Q_discharged_1MPa = sol_DFN_1["Discharge capacity [A.h]"].entries
Q_discharged_10MPa = sol_DFN_2["Discharge capacity [A.h]"].entries

Q_discharged_0Pa_C = sol_DFN_C["Discharge capacity [A.h]"].entries
Q_discharged_1MPa_C = sol_DFN_1_C["Discharge capacity [A.h]"].entries
Q_discharged_10MPa_C = sol_DFN_2_C["Discharge capacity [A.h]"].entries

x_val = [
        # Q_discharged_0Pa,
        # Q_discharged_1MPa,
        # Q_discharged_10MPa,
        Q_discharged_0Pa_C*(-1),
        Q_discharged_1MPa_C*(-1),
        Q_discharged_10MPa_C*(-1),        
         ]

y_val = [
        # sol_DFN["Battery voltage [V]"].entries, 
        # sol_DFN_1["Battery voltage [V]"].entries,
        # sol_DFN_2["Battery voltage [V]"].entries,
        sol_DFN_C["Battery voltage [V]"].entries, 
        sol_DFN_1_C["Battery voltage [V]"].entries,
        sol_DFN_2_C["Battery voltage [V]"].entries,
         ]


plt.figure(figsize=fig_size)

for i in range(len(x_val)):
    plt.plot(x_val[i], y_val[i], label=[fr"$\sigma_{{\mathrm{{h}}}}$ = 0 Pa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = 1 MPa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = 10 MPa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = 0 Pa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = 1 MPa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = 10 MPa",
                                        ][i],
                                linestyle=['-', '--', '-', '-', '--', '-' ][i]

                                        )
    
# # draw line at soc = 0.5
# plt.axvline(x=0.5, color='gray', linestyle='--', label='SOC = 0.5')

# # draw line at 3.7156 V
# plt.axhline(y=3.7156, color='gray', linestyle='--', label='V = 3.7156 V')

plt.xlabel("Q [Ah]")
plt.ylabel("V [V]")
# plt.legend(loc="upper right", bbox_to_anchor=(1.5, 0.9))
plt.legend()
# plt.grid()

# save to eps
# plt.savefig("NMC_LiC6_VQ_charge.eps", format='eps',dpi= 300, bbox_inches='tight')

## ICA

In [ ]:
dQ_dV = 1/savgol_filter(np.gradient(sol_DFN["Voltage [V]"].entries [70:], DOD[70:]),10,2)
dQ_dV_1 = 1/savgol_filter(np.gradient(sol_DFN_1["Voltage [V]"].entries[75:],DOD_1 [75:]),10,2)
dQ_dV_2 = 1/savgol_filter(np.gradient(sol_DFN_2["Voltage [V]"].entries[80:],DOD_2 [80:]),10,2)

dQ_dV_C = 1/savgol_filter(np.gradient(sol_DFN_C["Voltage [V]"].entries [102:], DOD_C[102:]),10,2)
dQ_dV_1_C = 1/savgol_filter(np.gradient(sol_DFN_1_C["Voltage [V]"].entries[111:],DOD_1_C [111:]),10,2)
dQ_dV_2_C = 1/savgol_filter(np.gradient(sol_DFN_2_C["Voltage [V]"].entries[122:],DOD_2_C [122:]),10,2)


# plot
x_val = [sol_DFN["Voltage [V]"].entries[70:],
        sol_DFN_1["Voltage [V]"].entries[75:],
        sol_DFN_2["Voltage [V]"].entries[80:],
         ]

y_val = [dQ_dV,
        dQ_dV_1,
        dQ_dV_2,
         ]

x_val_C = [sol_DFN_C["Voltage [V]"].entries[102:],
        sol_DFN_1_C["Voltage [V]"].entries[111:],
        sol_DFN_2_C["Voltage [V]"].entries[122:],
         ]

y_val_C = [dQ_dV_C,
        dQ_dV_1_C,
        dQ_dV_2_C,
         ]


plt.figure(figsize=fig_size)
cmap = plt.get_cmap('tab10')
for i in range(len(x_val)):
    color = cmap(i % 10)
    plt.plot(x_val[i], -y_val[i], label=["$\sigma_{\mathrm{h}}=$0 Pa", 
                                        "$\sigma_{\mathrm{h}}=$1 MPa", 
                                        "$\sigma_{\mathrm{h}}=$10 MPa", 

                                                                                ][i], 
                                        linestyle=["-", "--", "-", ][i], color=color)
for i in range(len(x_val_C)):
    color = cmap(i % 6)
    plt.plot(x_val_C[i], -y_val_C[i],linestyle=["-", "--", "-", "-", "-"][i],  color=color)
    
# plt.xlim(4.15, 4.154)
# plt.ylim(-1e6, 2e6)
plt.xlabel("V [V]")
plt.ylabel("dQ/dV [As/V]")
# plt.grid()
# plt.legend(loc="center right", bbox_to_anchor=(1.4, 0.5))
plt.legend()

# save to eps
# plt.savefig("NMC_LiC6_dQdV.eps", format='eps',dpi= 300, bbox_inches='tight')


## EIS

In [ ]:
Z_0Pa_soc30_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_0Pa_soc30_omgE4_lowF_Clerici.npy") 
Z_1e6Pa_soc30_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_1e6Pa_soc30_omgE4_lowF_Clerici.npy")
Z_1e7Pa_soc30_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_1e7Pa_soc30_omgE4_lowF_Clerici.npy")

plt.plot(Z_0Pa_soc30_omgE4_Clerici[:, 1], -Z_0Pa_soc30_omgE4_Clerici[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = 0 Pa")

plt.plot(Z_1e6Pa_soc30_omgE4_Clerici[:, 1], -Z_1e6Pa_soc30_omgE4_Clerici[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = 1 MPa")

plt.plot(Z_1e7Pa_soc30_omgE4_Clerici[:, 1], -Z_1e7Pa_soc30_omgE4_Clerici[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = 10 MPa")


# Labels and title
# Labels and title
plt.xlabel('Re(Z) [Ω]')
plt.ylabel('-Im(Z) [Ω]')

# plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))
plt.xlim(0, 0.020)
plt.ylim(0, 0.020)
plt.title('V = 3.64 V')
# plt.grid()

# save figure as eps
# plt.savefig('NMC_LiC6_Nyquist_soc30_omgE4_lowF_Clerici.eps', format='eps', dpi = 300, bbox_inches='tight')


In [ ]:
Z_0Pa_soc50_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_0Pa_soc50_omgE4_lowF_Clerici.npy") 
Z_1e6Pa_soc50_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_1e6Pa_soc50_omgE4_lowF_Clerici.npy")
Z_1e7Pa_soc50_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_1e7Pa_soc50_omgE4_lowF_Clerici.npy")


plt.plot(Z_0Pa_soc50_omgE4_Clerici[:, 1], -Z_0Pa_soc50_omgE4_Clerici[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = 0 Pa")

plt.plot(Z_1e6Pa_soc50_omgE4_Clerici[:, 1], -Z_1e6Pa_soc50_omgE4_Clerici[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = 1 MPa")

plt.plot(Z_1e7Pa_soc50_omgE4_Clerici[:, 1], -Z_1e7Pa_soc50_omgE4_Clerici[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = 10 MPa")


# Labels and title
plt.xlabel('Re(Z) [Ω]')
plt.ylabel('-Im(Z) [Ω]')

# plt.legend(loc='center right', bbox_to_anchor=(1.6, 0.5))
# plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))
plt.title('V = 3.72 V')
plt.xlim(0, 0.020)
plt.ylim(0, 0.020)
# plt.grid()

# save figure as eps
# plt.savefig('NMC_LiC6_Nyquist_soc50_omgE4_lowF_Clerici.eps', format='eps', dpi = 300, bbox_inches='tight')


In [ ]:
Z_0Pa_soc80_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_0Pa_soc80_omgE4_lowF_Clerici.npy") 
Z_1e6Pa_soc80_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_1e6Pa_soc80_omgE4_lowF_Clerici.npy")
Z_1e7Pa_soc80_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_1e7Pa_soc80_omgE4_lowF_Clerici.npy")

plt.plot(Z_0Pa_soc80_omgE4_Clerici[:, 1], -Z_0Pa_soc80_omgE4_Clerici[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = 0 Pa")

plt.plot(Z_1e6Pa_soc80_omgE4_Clerici[:, 1], -Z_1e6Pa_soc80_omgE4_Clerici[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = 1 MPa")

plt.plot(Z_1e7Pa_soc80_omgE4_Clerici[:, 1], -Z_1e7Pa_soc80_omgE4_Clerici[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = 10 MPa")

# Labels and title
plt.xlabel('Re(Z) [Ω]')
plt.ylabel('-Im(Z) [Ω]')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))
plt.title('V= 3.98 V')
plt.xlim(0, 0.020)
plt.ylim(0, 0.020)
# plt.grid()

# save figure as eps
# plt.savefig('NMC_LiC6_Nyquist_soc80_omgE4_lowF_Clerici.eps', format='eps', dpi = 300, bbox_inches='tight')


## DRT

In [ ]:
# read from txt
Z_0Pa_soc30_omgE4 = np.loadtxt("NMC_LiC6 results/NMC_LiC6_Z_0Pa_soc30_omgE4_Clerici_DRT.txt")
Z_0Pa_soc50_omgE4 = np.loadtxt("NMC_LiC6 results/NMC_LiC6_Z_0Pa_soc50_omgE4_Clerici_DRT.txt")
Z_0Pa_soc80_omgE4 = np.loadtxt("NMC_LiC6 results/NMC_LiC6_Z_0Pa_soc80_omgE4_Clerici_DRT.txt")

Z_1e6Pa_soc30_omgE4 = np.loadtxt("NMC_LiC6 results/NMC_LiC6_Z_1e6Pa_soc30_omgE4_Clerici_DRT.txt")
Z_1e6Pa_soc50_omgE4 = np.loadtxt("NMC_LiC6 results/NMC_LiC6_Z_1e6Pa_soc50_omgE4_Clerici_DRT.txt")
Z_1e6Pa_soc80_omgE4 = np.loadtxt("NMC_LiC6 results/NMC_LiC6_Z_1e6Pa_soc80_omgE4_Clerici_DRT.txt")

Z_1e7Pa_soc30_omgE4 = np.loadtxt("NMC_LiC6 results/NMC_LiC6_Z_1e7Pa_soc30_omgE4_Clerici_DRT.txt")
Z_1e7Pa_soc50_omgE4 = np.loadtxt("NMC_LiC6 results/NMC_LiC6_Z_1e7Pa_soc50_omgE4_Clerici_DRT.txt")
Z_1e7Pa_soc80_omgE4 = np.loadtxt("NMC_LiC6 results/NMC_LiC6_Z_1e7Pa_soc80_omgE4_Clerici_DRT.txt")

In [ ]:
# Plot DRT results
plt.plot(Z_0Pa_soc30_omgE4[:, 0], Z_0Pa_soc30_omgE4[:, 1], label='0 Pa')
plt.plot(Z_1e6Pa_soc30_omgE4[:, 0], Z_1e6Pa_soc30_omgE4[:, 1], label='1e6 Pa')
plt.plot(Z_1e7Pa_soc30_omgE4[:, 0], Z_1e7Pa_soc30_omgE4[:, 1], label='1e7 Pa')

# add logarithmic scale to x axis
plt.xscale('log')

# add labels and titlež
plt.xlabel(r'$\tau$ [s]')
plt.ylabel('$\gamma$  [/]')
plt.title('V = 3.64 V')
# plt.grid(True, which='both')
# plt.legend()

# save figure as eps
# plt.savefig('NMC_LiC6_DRT_soc30_omgE4.eps', format='eps', dpi=300, bbox_inches='tight')

In [ ]:
# Plot DRT results
plt.plot(Z_0Pa_soc50_omgE4[:, 0], Z_0Pa_soc50_omgE4[:, 1], label='0 Pa')
plt.plot(Z_1e6Pa_soc50_omgE4[:, 0], Z_1e6Pa_soc50_omgE4[:, 1], label='1e6 Pa')
plt.plot(Z_1e7Pa_soc50_omgE4[:, 0], Z_1e7Pa_soc50_omgE4[:, 1], label='1e7 Pa')

# add logarithmic scale to x axis
plt.xscale('log')

# add labels and titlež
plt.xlabel(r'$\tau$ [s]')
plt.ylabel('$\gamma$  [/]')
plt.title('V = 3.72 V')
# plt.grid(True, which='both')
# plt.legend()

# save as eps
# plt.savefig('NMC_LiC6_DRT_soc50_omgE4.eps', format='eps', dpi=300, bbox_inches='tight')

In [ ]:
# Plot DRT results
plt.plot(Z_0Pa_soc80_omgE4[:, 0], Z_0Pa_soc80_omgE4[:, 1], label='$\sigma_{{\mathrm{{h}}}}$ = 0 Pa')
plt.plot(Z_1e6Pa_soc80_omgE4[:, 0], Z_1e6Pa_soc80_omgE4[:, 1], label='$\sigma_{{\mathrm{{h}}}}$ = 1 MPa')
plt.plot(Z_1e7Pa_soc80_omgE4[:, 0], Z_1e7Pa_soc80_omgE4[:, 1], label='$\sigma_{{\mathrm{{h}}}}$ = 10 MPa')

# add logarithmic scale to x axis
plt.xscale('log')

# add labels and titlež
plt.xlabel(r'$\tau$ [s]')
plt.ylabel('$\gamma$  [/]')
plt.title('V = 3.98 V')
# plt.grid(True, which='both')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# save as eps
# plt.savefig('NMC_LiC6_DRT_soc80_omgE4.eps', format='eps', dpi=300, bbox_inches='tight')

# Notranja stanja

In [ ]:
# Formatiranje grafov
# Nastavitev velikosti grafa (širina, višina v inch-ih)
fig_size = figsize=(8, 6)  

# Nastavitev velikosti pisave
plt.rcParams.update({
    'font.size': 12,           # Osnovna velikost pisave
    'axes.titlesize': 16,      # Velikost naslova grafa
    'axes.labelsize': 14,      # Velikost oznak osi
    'xtick.labelsize': 12,     # Velikost oznak na x osi
    'ytick.labelsize': 12,     # Velikost oznak na y osi
    'legend.fontsize': 12,     # Velikost pisave v legendi
})

In [ ]:
time_frame = 178000

In [ ]:
# -----------------------
# Plotting results at last time step
# -----------------------
param_list = [param0, param1, param2]
sol_list = [sol_DFN, sol_DFN_1, sol_DFN_2]


# Colors and labels for each solution
colors = ['C0', 'C1', 'C2']
labels = ['0 Pa', '1 MPa', '10 MPa']

# Collect data from all solutions
all_data = []
for i, (param, sol) in enumerate(zip(param_list, sol_list)):
    c_ely   = sol["Electrolyte concentration [mol.m-3]"].entries[:, time_frame]
    phi_ely = sol['Electrolyte potential [V]'].data[:, time_frame]
    c_s_cathode = sol["Positive particle surface concentration [mol.m-3]"].data[:, time_frame]
    c_s_separator = np.zeros(var_pts['x_s'])
    c_s_anode = sol["Negative particle surface concentration [mol.m-3]"].data[:, time_frame]
    phi_s_cathode = sol['Positive electrode potential [V]'].data[:, time_frame]
    phi_s_separator = np.zeros(var_pts['x_s'])
    phi_s_anode = sol['Negative electrode potential [V]'].data[:, time_frame]

    # Build spatial coordinates
    x_coord = sol["x [m]"].entries[:, time_frame]
    x_coord_cathode = sol["x_p [m]"].entries[:, time_frame]
    x_coord_separator = sol["x_s [m]"].entries[:, time_frame]
    x_coord_anode = sol["x_n [m]"].entries[:, time_frame]

    all_data.append({
        'c_ely': c_ely,
        'phi_ely': phi_ely,
        'c_s_cathode': c_s_cathode,
        'c_s_separator': c_s_separator,
        'c_s_anode': c_s_anode,
        'phi_s_cathode': phi_s_cathode,
        'phi_s_separator': phi_s_separator,
        'phi_s_anode': phi_s_anode,
        'x_coord': x_coord,
        'x_coord_cathode': x_coord_cathode,
        'x_coord_separator': x_coord_separator,
        'x_coord_anode': x_coord_anode,
    })

# Create 3x2 figure for the six variables
fig, axs = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle(f"Spatial Profiles Comparison - Time Step: {time_frame} (C/50)", fontsize=14)

for i, data in enumerate(all_data):
    # Electrolyte concentration
    axs[0, 0].plot(data['x_coord']*1e6, data['c_ely'], color=colors[i], label=labels[i])
    axs[0, 0].axvspan(0, data['x_coord_separator'][0]*1e6, alpha=0.05, color='gray')
    axs[0, 0].axvspan(data['x_coord_separator'][-1]*1e6, data['x_coord_cathode'][-1]*1e6, alpha=0.05, color='orange')

    # Electrolyte potential
    axs[1, 0].plot(data['x_coord']*1e6, data['phi_ely'], color=colors[i])
    axs[1, 0].axvspan(0, data['x_coord_separator'][0]*1e6, alpha=0.05, color='gray')
    axs[1, 0].axvspan(data['x_coord_separator'][-1]*1e6, data['x_coord_cathode'][-1]*1e6, alpha=0.05, color='orange')


    # Anode solid surface concentration
    axs[0, 1].plot(data['x_coord_anode']*1e6, data['c_s_anode'], color=colors[i])

    # Anode solid potential
    axs[1, 1].plot(data['x_coord_anode']*1e6, data['phi_s_anode'], color=colors[i])

    # Cathode solid surface concentration
    axs[0, 2].plot(data['x_coord_cathode']*1e6, data['c_s_cathode'], color=colors[i])

    # Cathode solid potential
    axs[1, 2].plot(data['x_coord_cathode']*1e6, data['phi_s_cathode'], color=colors[i])

# Set titles, labels, grids
axs[0, 0].set_title("Electrolyte Concentration")
axs[0, 0].set_ylabel(r"$c_{ely}$ [mol.m$^{-3}$]")
axs[0, 0].set_xlabel("x [µm]")
axs[0, 0].grid(True)

axs[1, 0].set_title(r"Electrolyte Potential")
axs[1, 0].set_ylabel(r"$\phi_{ely}$ [V]")
axs[1, 0].set_xlabel("x [µm]")
axs[1, 0].grid(True)

axs[0, 1].set_title("Anode Solid Surface Concentration")
axs[0, 1].set_ylabel(r"$c_{s,surf}$ [mol.m$^{-3}$]")
axs[0, 1].set_xlabel("x [µm]")
axs[0, 1].grid(True)

axs[1, 1].set_title(r"Anode Solid Potential")
axs[1, 1].set_ylabel(r"$\phi_s$ [V]")
axs[1, 1].set_xlabel("x [µm]")
axs[1, 1].grid(True)

axs[0, 2].set_title("Cathode Solid Surface Concentration")
axs[0, 2].set_ylabel(r"$c_{s,surf}$ [mol.m$^{-3}$]")
axs[0, 2].set_xlabel("x [µm]")
axs[0, 2].grid(True)

axs[1, 2].set_title(r"Cathode Solid Potential")
axs[1, 2].set_ylabel(r"$\phi_s$ [V]")
axs[1, 2].set_xlabel("x [µm]")
axs[1, 2].grid(True)

# Legend (from first axis)
handles, labels_list = axs[0, 0].get_legend_handles_labels()
fig.legend(handles, labels_list, loc='upper center', bbox_to_anchor=(0.5, -0.02), ncol=3)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
# -----------------------
# Plotting results as animation
# -----------------------
param_list = [param0, param1, param2]
sol_list = [sol_DFN, sol_DFN_1, sol_DFN_2]

# Colors and labels for each solution
colors = ['C0', 'C1', 'C2']
labels = ['0 Pa', '1 MPa', '10 MPa']

# Get number of time steps
num_time_steps = sol_DFN_2["Time [s]"].entries.shape[0]-1

# Create 2x3 figure for the six variables
fig, axs = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle(f"Spatial Profiles Comparison - Time: 0.00 s (C/50)", fontsize=14)

# Initialize line objects for each subplot and each solution (6 subplots x 3 solutions = 18 lines)
lines_c_ely = [axs[0, 0].plot([], [], color=colors[i], label=labels[i])[0] for i in range(3)]
lines_phi_ely = [axs[1, 0].plot([], [], color=colors[i])[0] for i in range(3)]
lines_c_s_anode = [axs[0, 1].plot([], [], color=colors[i])[0] for i in range(3)]
lines_phi_s_anode = [axs[1, 1].plot([], [], color=colors[i])[0] for i in range(3)]
lines_c_s_cathode = [axs[0, 2].plot([], [], color=colors[i])[0] for i in range(3)]
lines_phi_s_cathode = [axs[1, 2].plot([], [], color=colors[i])[0] for i in range(3)]

# Set up axvspan backgrounds (once)
axs[0, 0].axvspan(0, sol_DFN["x_n [m]"].entries[-1, 0]*1e6, alpha=0.1, color='gray')
axs[0, 0].axvspan(sol_DFN["x_s [m]"].entries[-1, 0]*1e6, sol_DFN["x_p [m]"].entries[-1, 0]*1e6, alpha=0.1, color='orange')
axs[1, 0].axvspan(0, sol_DFN["x_n [m]"].entries[-1, 0]*1e6, alpha=0.1, color='gray')
axs[1, 0].axvspan(sol_DFN["x_s [m]"].entries[-1, 0]*1e6, sol_DFN["x_p [m]"].entries[-1, 0]*1e6, alpha=0.1, color='orange')


# Set titles, labels, grids
axs[0, 0].set_title("Electrolyte Concentration")
axs[0, 0].set_ylabel(r"$c_{ely}$ [mol.m$^{-3}$]")
axs[0, 0].set_xlabel("x [µm]")
axs[0, 0].grid(True)

axs[1, 0].set_title(r"Electrolyte Potential")
axs[1, 0].set_ylabel(r"$\phi_{ely}$ [V]")
axs[1, 0].set_xlabel("x [µm]")
axs[1, 0].grid(True)

axs[0, 1].set_title("Anode Solid Surface Concentration")
axs[0, 1].set_ylabel(r"$c_{s,surf}$ [mol.m$^{-3}$]")
axs[0, 1].set_xlabel("x [µm]")
axs[0, 1].grid(True)

axs[1, 1].set_title(r"Anode Solid Potential")
axs[1, 1].set_ylabel(r"$\phi_s$ [V]")
axs[1, 1].set_xlabel("x [µm]")
axs[1, 1].grid(True)

axs[0, 2].set_title("Cathode Solid Surface Concentration")
axs[0, 2].set_ylabel(r"$c_{s,surf}$ [mol.m$^{-3}$]")
axs[0, 2].set_xlabel("x [µm]")
axs[0, 2].grid(True)

axs[1, 2].set_title(r"Cathode Solid Potential")
axs[1, 2].set_ylabel(r"$\phi_s$ [V]")
axs[1, 2].set_xlabel("x [µm]")
axs[1, 2].grid(True)

# Set axis limits globally
axs[0, 0].set_xlim(0, sol_DFN["x [m]"].entries[:, 0][-1]*1e6)
axs[1, 0].set_xlim(0, sol_DFN["x [m]"].entries[:, 0][-1]*1e6)
axs[0, 1].set_xlim(0, sol_DFN["x_n [m]"].entries[:, 0][-1]*1e6)
axs[1, 1].set_xlim(0, sol_DFN["x_n [m]"].entries[:, 0][-1]*1e6)
axs[0, 2].set_xlim(sol_DFN["x_s [m]"].entries[:, 0][-1]*1e6, sol_DFN["x_p [m]"].entries[:, 0][-1]*1e6)
axs[1, 2].set_xlim(sol_DFN["x_s [m]"].entries[:, 0][-1]*1e6, sol_DFN["x_p [m]"].entries[:, 0][-1]*1e6)

# Electrolyte concentration
axs[0, 0].set_ylim(997.5, 1002.5)

# Electrolyte potential
axs[1, 0].set_ylim(-0.4, -0.05)

# Anode solid surface concentration
axs[0, 1].set_ylim(0, 32000)

# Anode solid potential
axs[1, 1].set_ylim(-0.05, 0.05)

# Cathode solid surface concentration
axs[0, 2].set_ylim(0, 32000 )

# Cathode solid potential
axs[1, 2].set_ylim(2.74, 4.2)

# Legend
handles, labels_list = axs[0, 0].get_legend_handles_labels()
fig.legend(handles, labels_list, loc='upper center', bbox_to_anchor=(0.5, -0.02), ncol=3)

# Update function for animation
def update(frame):
    time_idx = frame
    
    for sol_idx in range(3):
        sol = sol_list[sol_idx]
        
        # Extract data at current time frame
        c_ely = sol["Electrolyte concentration [mol.m-3]"].entries[:, time_idx]
        phi_ely = sol['Electrolyte potential [V]'].data[:, time_idx]
        c_s_anode = sol["Negative particle surface concentration [mol.m-3]"].data[:, time_idx]
        phi_s_anode = sol['Negative electrode potential [V]'].data[:, time_idx]
        c_s_cathode = sol["Positive particle surface concentration [mol.m-3]"].data[:, time_idx]
        phi_s_cathode = sol['Positive electrode potential [V]'].data[:, time_idx]
        
        # Get coordinates
        x_coord = sol["x [m]"].entries[:, time_idx]
        x_coord_anode = sol["x_n [m]"].entries[:, time_idx]
        x_coord_cathode = sol["x_p [m]"].entries[:, time_idx]
        
        # Update line data
        lines_c_ely[sol_idx].set_data(x_coord*1e6, c_ely)
        lines_phi_ely[sol_idx].set_data(x_coord*1e6, phi_ely)
        lines_c_s_anode[sol_idx].set_data(x_coord_anode*1e6, c_s_anode)
        lines_phi_s_anode[sol_idx].set_data(x_coord_anode*1e6, phi_s_anode)
        lines_c_s_cathode[sol_idx].set_data(x_coord_cathode*1e6, c_s_cathode)
        lines_phi_s_cathode[sol_idx].set_data(x_coord_cathode*1e6, phi_s_cathode)
    
    # Update title with current time
    time_val = sol_DFN["Time [s]"].entries[time_idx]
    fig.suptitle(f"Spatial Profiles Comparison - Time: {time_val:.2f} s (C/50)", fontsize=14)
    
    # Return all line objects
    all_lines = lines_c_ely + lines_phi_ely + lines_c_s_anode + lines_phi_s_anode + lines_c_s_cathode + lines_phi_s_cathode
    return all_lines

plt.tight_layout(rect=[0, 0.03, 1, 0.95])

# Create animation
# ani = FuncAnimation(fig, update, frames=range(0,num_time_steps, 1), blit=True, interval=100)
ani = FuncAnimation(fig, update, frames=range(0, num_time_steps, 10000), blit=True, interval=100)
plt.close()

# Display animation
HTML(ani.to_jshtml())

SOC 100%

In [ ]:
# -----------------------
# Plotting results at last time step
# -----------------------
param_list = [param0, param1, param2]
sol_list = [sol_DFN, sol_DFN_1, sol_DFN_2]
time_frame = 1

# Create a single figure with 2x2 subplots
fig, axs = plt.subplots(2, 2, figsize=(14, 10))
# fig.suptitle(f"Spatial Profiles Comparison - Time Step: {time_frame} (C/50)", fontsize=14)

# Colors and labels for each solution
colors = ['C0', 'C1', 'C2']
labels = ['0 Pa', '1 MPa', '10 MPa']

# Collect data from all solutions
all_data = []
for i, (param, sol) in enumerate(zip(param_list, sol_list)):
    c_ely   = sol["Electrolyte concentration [mol.m-3]"].entries[:, time_frame]
    phi_ely = sol['Electrolyte potential [V]'].data[:, time_frame]
    c_s_cathode     = sol["Positive particle surface concentration [mol.m-3]"].data[:, time_frame]
    c_s_separator     = np.zeros(var_pts['x_s'])
    c_s_anode     = sol["Negative particle surface concentration [mol.m-3]"].data[:, time_frame]
    phi_s_cathode   = sol['Positive electrode potential [V]'].data[:, time_frame]
    phi_s_separator   = np.zeros(var_pts['x_s'])
    phi_s_anode     = sol['Negative electrode potential [V]'].data[:, time_frame]

    # Build spatial coordinates
    x_coord = sol["x [m]"].entries[:, time_frame]
    x_coord_cathode = sol["x_p [m]"].entries[:, time_frame]
    x_coord_separator = sol["x_s [m]"].entries[:, time_frame]
    x_coord_anode = sol["x_n [m]"].entries[:, time_frame]

    all_data.append({
        'c_ely': c_ely,
        'phi_ely': phi_ely,
        'c_s_cathode': c_s_cathode,
        'c_s_separator': c_s_separator,
        'c_s_anode': c_s_anode,
        'phi_s_cathode': phi_s_cathode,
        'phi_s_separator': phi_s_separator,
        'phi_s_anode': phi_s_anode,
        'x_coord': x_coord,
        'x_coord_cathode': x_coord_cathode,
        'x_coord_separator': x_coord_separator,
        'x_coord_anode': x_coord_anode,
    })

# Plot all solutions on each subplot
for i, data in enumerate(all_data):
    # Plot 0,0: Electrolyte Concentration
    axs[0, 0].plot(data['x_coord']*1e6, data['c_ely'], color=colors[i], label=labels[i])
    axs[0, 0].axvspan(0, data['x_coord_separator'][0]*1e6, alpha=0.05, color='gray')
    axs[0, 0].axvspan(data['x_coord_separator'][-1]*1e6, data['x_coord_cathode'][-1]*1e6, alpha=0.05, color='orange')
    axs[0, 0].set_title("Electrolyte Concentration")
    axs[0, 0].set_ylabel(r"$c_{ely}$ [mol.m$^{-3}$]")
    axs[0, 0].set_xlabel("x [µm]")
    axs[0, 0].grid(True)

    # Plot 0,1: Solid Surface Concentration
    axs[0, 1].plot(data['x_coord_cathode']*1e6, data['c_s_cathode'], color=colors[i], label=labels[i])
    axs[0, 1].plot(data['x_coord_separator']*1e6, data['c_s_separator'], color=colors[i])
    axs[0, 1].plot(data['x_coord_anode']*1e6, data['c_s_anode'], color=colors[i])
    axs[0, 1].axvspan(0, data['x_coord_separator'][0]*1e6, alpha=0.05, color='gray')
    axs[0, 1].axvspan(data['x_coord_separator'][-1]*1e6, data['x_coord_cathode'][-1]*1e6, alpha=0.05, color='orange')
    axs[0, 1].set_ylabel(r"$c_{s,surf}$ [mol.m$^{-3}$]")
    axs[0, 1].set_title("Solid Surface Concentration")
    axs[0, 1].set_xlabel("x [µm]")
    axs[0, 1].grid(True)

    # Plot 1,0: Electrolyte Potential
    axs[1, 0].plot(data['x_coord']*1e6, data['phi_ely'], color=colors[i], label=labels[i])
    axs[1, 0].axvspan(0, data['x_coord_separator'][0]*1e6, alpha=0.05, color='gray')
    axs[1, 0].axvspan(data['x_coord_separator'][-1]*1e6, data['x_coord_cathode'][-1]*1e6, alpha=0.05, color='orange')
    axs[1, 0].set_ylabel(r"$\phi_{ely}$ [V]")
    axs[1, 0].set_title(r"Electrolyte Potential")
    axs[1, 0].set_xlabel("x [µm]")
    axs[1, 0].grid(True)

    # Plot 1,1: Solid Potential
    axs[1, 1].plot(data['x_coord_cathode']*1e6, data['phi_s_cathode'], color=colors[i], label=labels[i])
    axs[1, 1].plot(data['x_coord_separator']*1e6, data['phi_s_separator'], color=colors[i])
    axs[1, 1].plot(data['x_coord_anode']*1e6, data['phi_s_anode'], color=colors[i])
    if i == 0:
        axs[1, 1].axvspan(0, data['x_coord_separator'][0]*1e6, alpha=0.05, color='gray', label='Anode')
        axs[1, 1].axvspan(data['x_coord_separator'][-1]*1e6, data['x_coord_cathode'][-1]*1e6, alpha=0.05, color='orange', label='Cathode')
    else:
        axs[1, 1].axvspan(0, data['x_coord_separator'][0]*1e6, alpha=0.05, color='gray', label='_nolegend_')
        axs[1, 1].axvspan(data['x_coord_separator'][-1]*1e6, data['x_coord_cathode'][-1]*1e6, alpha=0.05, color='orange', label='_nolegend_')
    axs[1, 1].set_ylabel(r"$\phi_s$ [V]")
    axs[1, 1].set_title(r"Solid Potential")
    axs[1, 1].set_xlabel("x [µm]")
    axs[1, 1].grid(True)

    
    

# Add legend
# handles, labels_list = axs[1, 1].get_legend_handles_labels()
# fig.legend(handles, labels_list, loc='upper center', bbox_to_anchor=(0.5, -0.02), ncol=3)
handles, labels_list = axs[1, 1].get_legend_handles_labels()
special = ["Cathode", "10 MPa"]
normal_h, normal_l, special_h, special_l = [], [], [], []
for h, l in zip(handles, labels_list):
    if l in special:
        special_h.append(h); special_l.append(l)
    else:
        normal_h.append(h); normal_l.append(l)
handles_ordered = normal_h + special_h
labels_ordered = normal_l + special_l
fig.legend(handles_ordered, labels_ordered, loc='upper center', bbox_to_anchor=(0.5, -0.02), ncol=3)

plt.tight_layout(rect=[0, 0.03, 1, 0.98])
plt.show()

SOC 80%

In [ ]:
Q = sol_DFN["Discharge capacity [A.h]"].entries
t = sol_DFN["Time [s]"].entries

Q_half = Q[-1] * 0.2

# nearest time point
idx = np.argmin(np.abs(Q - Q_half))
time_half = t[idx]

print("Time at 80% capacity:", time_half, "s")
print("Capacity at that time:", Q[idx])

In [ ]:
# -----------------------
# Plotting results at last time step
# -----------------------
param_list = [param0, param1, param2]
sol_list = [sol_DFN, sol_DFN_1, sol_DFN_2]
time_frame = 35795 # SOC 80%

# Create a single figure with 2x2 subplots
fig, axs = plt.subplots(2, 2, figsize=(14, 10))
# fig.suptitle(f"Spatial Profiles Comparison - Time Step: {time_frame} (C/50)", fontsize=14)

# Colors and labels for each solution
colors = ['C0', 'C1', 'C2']
labels = ['0 Pa', '1 MPa', '10 MPa']

# Collect data from all solutions
all_data = []
for i, (param, sol) in enumerate(zip(param_list, sol_list)):
    c_ely   = sol["Electrolyte concentration [mol.m-3]"].entries[:, time_frame]
    phi_ely = sol['Electrolyte potential [V]'].data[:, time_frame]
    c_s_cathode     = sol["Positive particle surface concentration [mol.m-3]"].data[:, time_frame]
    c_s_separator     = np.zeros(var_pts['x_s'])
    c_s_anode     = sol["Negative particle surface concentration [mol.m-3]"].data[:, time_frame]
    phi_s_cathode   = sol['Positive electrode potential [V]'].data[:, time_frame]
    phi_s_separator   = np.zeros(var_pts['x_s'])
    phi_s_anode     = sol['Negative electrode potential [V]'].data[:, time_frame]

    # Build spatial coordinates
    x_coord = sol["x [m]"].entries[:, time_frame]
    x_coord_cathode = sol["x_p [m]"].entries[:, time_frame]
    x_coord_separator = sol["x_s [m]"].entries[:, time_frame]
    x_coord_anode = sol["x_n [m]"].entries[:, time_frame]

    all_data.append({
        'c_ely': c_ely,
        'phi_ely': phi_ely,
        'c_s_cathode': c_s_cathode,
        'c_s_separator': c_s_separator,
        'c_s_anode': c_s_anode,
        'phi_s_cathode': phi_s_cathode,
        'phi_s_separator': phi_s_separator,
        'phi_s_anode': phi_s_anode,
        'x_coord': x_coord,
        'x_coord_cathode': x_coord_cathode,
        'x_coord_separator': x_coord_separator,
        'x_coord_anode': x_coord_anode,
    })

# Plot all solutions on each subplot
for i, data in enumerate(all_data):
    # Plot 0,0: Electrolyte Concentration
    axs[0, 0].plot(data['x_coord']*1e6, data['c_ely'], color=colors[i], label=labels[i])
    axs[0, 0].axvspan(0, data['x_coord_separator'][0]*1e6, alpha=0.05, color='gray')
    axs[0, 0].axvspan(data['x_coord_separator'][-1]*1e6, data['x_coord_cathode'][-1]*1e6, alpha=0.05, color='orange')
    axs[0, 0].set_title("Electrolyte Concentration")
    axs[0, 0].set_ylabel(r"$c_{ely}$ [mol.m$^{-3}$]")
    axs[0, 0].set_xlabel("x [µm]")
    axs[0, 0].grid(True)

    # Plot 0,1: Solid Surface Concentration
    axs[0, 1].plot(data['x_coord_cathode']*1e6, data['c_s_cathode'], color=colors[i], label=labels[i])
    axs[0, 1].plot(data['x_coord_separator']*1e6, data['c_s_separator'], color=colors[i])
    axs[0, 1].plot(data['x_coord_anode']*1e6, data['c_s_anode'], color=colors[i])
    axs[0, 1].axvspan(0, data['x_coord_separator'][0]*1e6, alpha=0.05, color='gray')
    axs[0, 1].axvspan(data['x_coord_separator'][-1]*1e6, data['x_coord_cathode'][-1]*1e6, alpha=0.05, color='orange')
    axs[0, 1].set_ylabel(r"$c_{s,surf}$ [mol.m$^{-3}$]")
    axs[0, 1].set_title("Solid Surface Concentration")
    axs[0, 1].set_xlabel("x [µm]")
    axs[0, 1].grid(True)

    # Plot 1,0: Electrolyte Potential
    axs[1, 0].plot(data['x_coord']*1e6, data['phi_ely'], color=colors[i], label=labels[i])
    axs[1, 0].axvspan(0, data['x_coord_separator'][0]*1e6, alpha=0.05, color='gray')
    axs[1, 0].axvspan(data['x_coord_separator'][-1]*1e6, data['x_coord_cathode'][-1]*1e6, alpha=0.05, color='orange')
    axs[1, 0].set_ylabel(r"$\phi_{ely}$ [V]")
    axs[1, 0].set_title(r"Electrolyte Potential")
    axs[1, 0].set_xlabel("x [µm]")
    axs[1, 0].grid(True)

    # Plot 1,1: Solid Potential
    axs[1, 1].plot(data['x_coord_cathode']*1e6, data['phi_s_cathode'], color=colors[i], label=labels[i])
    axs[1, 1].plot(data['x_coord_separator']*1e6, data['phi_s_separator'], color=colors[i])
    axs[1, 1].plot(data['x_coord_anode']*1e6, data['phi_s_anode'], color=colors[i])
    if i == 0:
        axs[1, 1].axvspan(0, data['x_coord_separator'][0]*1e6, alpha=0.05, color='gray', label='Anode')
        axs[1, 1].axvspan(data['x_coord_separator'][-1]*1e6, data['x_coord_cathode'][-1]*1e6, alpha=0.05, color='orange', label='Cathode')
    else:
        axs[1, 1].axvspan(0, data['x_coord_separator'][0]*1e6, alpha=0.05, color='gray', label='_nolegend_')
        axs[1, 1].axvspan(data['x_coord_separator'][-1]*1e6, data['x_coord_cathode'][-1]*1e6, alpha=0.05, color='orange', label='_nolegend_')
    axs[1, 1].set_ylabel(r"$\phi_s$ [V]")
    axs[1, 1].set_title(r"Solid Potential")
    axs[1, 1].set_xlabel("x [µm]")
    axs[1, 1].grid(True)

    
    

# Add legend
# handles, labels_list = axs[1, 1].get_legend_handles_labels()
# fig.legend(handles, labels_list, loc='upper center', bbox_to_anchor=(0.5, -0.02), ncol=3)
handles, labels_list = axs[1, 1].get_legend_handles_labels()
special = ["Cathode", "10 MPa"]
normal_h, normal_l, special_h, special_l = [], [], [], []
for h, l in zip(handles, labels_list):
    if l in special:
        special_h.append(h); special_l.append(l)
    else:
        normal_h.append(h); normal_l.append(l)
handles_ordered = normal_h + special_h
labels_ordered = normal_l + special_l
fig.legend(handles_ordered, labels_ordered, loc='upper center', bbox_to_anchor=(0.5, -0.02), ncol=3)

plt.tight_layout(rect=[0, 0.03, 1, 0.98])
plt.show()

SOC 50%

In [ ]:
Q = sol_DFN["Discharge capacity [A.h]"].entries
t = sol_DFN["Time [s]"].entries

Q_half = Q[-1] / 2

# nearest time point
idx = np.argmin(np.abs(Q - Q_half))
time_half = t[idx]

print("Time at half capacity:", time_half, "s")
print("Capacity at that time:", Q[idx])

In [ ]:
# -----------------------
# Plotting results at last time step
# -----------------------
param_list = [param0, param1, param2]
sol_list = [sol_DFN, sol_DFN_1, sol_DFN_2]
time_frame = 89488 # SOC 50%

# Create a single figure with 2x2 subplots
fig, axs = plt.subplots(2, 2, figsize=(14, 10))
# fig.suptitle(f"Spatial Profiles Comparison - Time Step: {time_frame} (C/50)", fontsize=14)

# Colors and labels for each solution
colors = ['C0', 'C1', 'C2']
labels = ['0 Pa', '1 MPa', '10 MPa']

# Collect data from all solutions
all_data = []
for i, (param, sol) in enumerate(zip(param_list, sol_list)):
    c_ely   = sol["Electrolyte concentration [mol.m-3]"].entries[:, time_frame]
    phi_ely = sol['Electrolyte potential [V]'].data[:, time_frame]
    c_s_cathode     = sol["Positive particle surface concentration [mol.m-3]"].data[:, time_frame]
    c_s_separator     = np.zeros(var_pts['x_s'])
    c_s_anode     = sol["Negative particle surface concentration [mol.m-3]"].data[:, time_frame]
    phi_s_cathode   = sol['Positive electrode potential [V]'].data[:, time_frame]
    phi_s_separator   = np.zeros(var_pts['x_s'])
    phi_s_anode     = sol['Negative electrode potential [V]'].data[:, time_frame]

    # Build spatial coordinates
    x_coord = sol["x [m]"].entries[:, time_frame]
    x_coord_cathode = sol["x_p [m]"].entries[:, time_frame]
    x_coord_separator = sol["x_s [m]"].entries[:, time_frame]
    x_coord_anode = sol["x_n [m]"].entries[:, time_frame]

    all_data.append({
        'c_ely': c_ely,
        'phi_ely': phi_ely,
        'c_s_cathode': c_s_cathode,
        'c_s_separator': c_s_separator,
        'c_s_anode': c_s_anode,
        'phi_s_cathode': phi_s_cathode,
        'phi_s_separator': phi_s_separator,
        'phi_s_anode': phi_s_anode,
        'x_coord': x_coord,
        'x_coord_cathode': x_coord_cathode,
        'x_coord_separator': x_coord_separator,
        'x_coord_anode': x_coord_anode,
    })

# Plot all solutions on each subplot
for i, data in enumerate(all_data):
    # Plot 0,0: Electrolyte Concentration
    axs[0, 0].plot(data['x_coord']*1e6, data['c_ely'], color=colors[i], label=labels[i])
    axs[0, 0].axvspan(0, data['x_coord_separator'][0]*1e6, alpha=0.05, color='gray')
    axs[0, 0].axvspan(data['x_coord_separator'][-1]*1e6, data['x_coord_cathode'][-1]*1e6, alpha=0.05, color='orange')
    axs[0, 0].set_title("Electrolyte Concentration")
    axs[0, 0].set_ylabel(r"$c_{ely}$ [mol.m$^{-3}$]")
    axs[0, 0].set_xlabel("x [µm]")
    axs[0, 0].grid(True)

    # Plot 0,1: Solid Surface Concentration
    axs[0, 1].plot(data['x_coord_cathode']*1e6, data['c_s_cathode'], color=colors[i], label=labels[i])
    axs[0, 1].plot(data['x_coord_separator']*1e6, data['c_s_separator'], color=colors[i])
    axs[0, 1].plot(data['x_coord_anode']*1e6, data['c_s_anode'], color=colors[i])
    axs[0, 1].axvspan(0, data['x_coord_separator'][0]*1e6, alpha=0.05, color='gray')
    axs[0, 1].axvspan(data['x_coord_separator'][-1]*1e6, data['x_coord_cathode'][-1]*1e6, alpha=0.05, color='orange')
    axs[0, 1].set_ylabel(r"$c_{s,surf}$ [mol.m$^{-3}$]")
    axs[0, 1].set_title("Solid Surface Concentration")
    axs[0, 1].set_xlabel("x [µm]")
    axs[0, 1].grid(True)

    # Plot 1,0: Electrolyte Potential
    axs[1, 0].plot(data['x_coord']*1e6, data['phi_ely'], color=colors[i], label=labels[i])
    axs[1, 0].axvspan(0, data['x_coord_separator'][0]*1e6, alpha=0.05, color='gray')
    axs[1, 0].axvspan(data['x_coord_separator'][-1]*1e6, data['x_coord_cathode'][-1]*1e6, alpha=0.05, color='orange')
    axs[1, 0].set_ylabel(r"$\phi_{ely}$ [V]")
    axs[1, 0].set_title(r"Electrolyte Potential")
    axs[1, 0].set_xlabel("x [µm]")
    axs[1, 0].grid(True)

    # Plot 1,1: Solid Potential
    axs[1, 1].plot(data['x_coord_cathode']*1e6, data['phi_s_cathode'], color=colors[i], label=labels[i])
    axs[1, 1].plot(data['x_coord_separator']*1e6, data['phi_s_separator'], color=colors[i])
    axs[1, 1].plot(data['x_coord_anode']*1e6, data['phi_s_anode'], color=colors[i])
    if i == 0:
        axs[1, 1].axvspan(0, data['x_coord_separator'][0]*1e6, alpha=0.05, color='gray', label='Anode')
        axs[1, 1].axvspan(data['x_coord_separator'][-1]*1e6, data['x_coord_cathode'][-1]*1e6, alpha=0.05, color='orange', label='Cathode')
    else:
        axs[1, 1].axvspan(0, data['x_coord_separator'][0]*1e6, alpha=0.05, color='gray', label='_nolegend_')
        axs[1, 1].axvspan(data['x_coord_separator'][-1]*1e6, data['x_coord_cathode'][-1]*1e6, alpha=0.05, color='orange', label='_nolegend_')
    axs[1, 1].set_ylabel(r"$\phi_s$ [V]")
    axs[1, 1].set_title(r"Solid Potential")
    axs[1, 1].set_xlabel("x [µm]")
    axs[1, 1].grid(True)

    
    

# Add legend
# handles, labels_list = axs[1, 1].get_legend_handles_labels()
# fig.legend(handles, labels_list, loc='upper center', bbox_to_anchor=(0.5, -0.02), ncol=3)
handles, labels_list = axs[1, 1].get_legend_handles_labels()
special = ["Cathode", "10 MPa"]
normal_h, normal_l, special_h, special_l = [], [], [], []
for h, l in zip(handles, labels_list):
    if l in special:
        special_h.append(h); special_l.append(l)
    else:
        normal_h.append(h); normal_l.append(l)
handles_ordered = normal_h + special_h
labels_ordered = normal_l + special_l
fig.legend(handles_ordered, labels_ordered, loc='upper center', bbox_to_anchor=(0.5, -0.02), ncol=3)

plt.tight_layout(rect=[0, 0.03, 1, 0.98])
plt.show()

SOC 30%

In [ ]:
Q = sol_DFN["Discharge capacity [A.h]"].entries
t = sol_DFN["Time [s]"].entries

Q_half = Q[-1] * 0.7

# nearest time point
idx = np.argmin(np.abs(Q - Q_half))
time_half = t[idx]

print("Time at 30% capacity:", time_half, "s")
print("Capacity at that time:", Q[idx])

In [ ]:
# -----------------------
# Plotting results at last time step
# -----------------------
param_list = [param0, param1, param2]
sol_list = [sol_DFN, sol_DFN_1, sol_DFN_2]
time_frame = 125283 # SOC 30%

# Create a single figure with 2x2 subplots
fig, axs = plt.subplots(2, 2, figsize=(14, 10))
# fig.suptitle(f"Spatial Profiles Comparison - Time Step: {time_frame} (C/50)", fontsize=14)

# Colors and labels for each solution
colors = ['C0', 'C1', 'C2']
labels = ['0 Pa', '1 MPa', '10 MPa']

# Collect data from all solutions
all_data = []
for i, (param, sol) in enumerate(zip(param_list, sol_list)):
    c_ely   = sol["Electrolyte concentration [mol.m-3]"].entries[:, time_frame]
    phi_ely = sol['Electrolyte potential [V]'].data[:, time_frame]
    c_s_cathode     = sol["Positive particle surface concentration [mol.m-3]"].data[:, time_frame]
    c_s_separator     = np.zeros(var_pts['x_s'])
    c_s_anode     = sol["Negative particle surface concentration [mol.m-3]"].data[:, time_frame]
    phi_s_cathode   = sol['Positive electrode potential [V]'].data[:, time_frame]
    phi_s_separator   = np.zeros(var_pts['x_s'])
    phi_s_anode     = sol['Negative electrode potential [V]'].data[:, time_frame]

    # Build spatial coordinates
    x_coord = sol["x [m]"].entries[:, time_frame]
    x_coord_cathode = sol["x_p [m]"].entries[:, time_frame]
    x_coord_separator = sol["x_s [m]"].entries[:, time_frame]
    x_coord_anode = sol["x_n [m]"].entries[:, time_frame]

    all_data.append({
        'c_ely': c_ely,
        'phi_ely': phi_ely,
        'c_s_cathode': c_s_cathode,
        'c_s_separator': c_s_separator,
        'c_s_anode': c_s_anode,
        'phi_s_cathode': phi_s_cathode,
        'phi_s_separator': phi_s_separator,
        'phi_s_anode': phi_s_anode,
        'x_coord': x_coord,
        'x_coord_cathode': x_coord_cathode,
        'x_coord_separator': x_coord_separator,
        'x_coord_anode': x_coord_anode,
    })

# Plot all solutions on each subplot
for i, data in enumerate(all_data):
    # Plot 0,0: Electrolyte Concentration
    axs[0, 0].plot(data['x_coord']*1e6, data['c_ely'], color=colors[i], label=labels[i])
    axs[0, 0].axvspan(0, data['x_coord_separator'][0]*1e6, alpha=0.05, color='gray')
    axs[0, 0].axvspan(data['x_coord_separator'][-1]*1e6, data['x_coord_cathode'][-1]*1e6, alpha=0.05, color='orange')
    axs[0, 0].set_title("Electrolyte Concentration")
    axs[0, 0].set_ylabel(r"$c_{ely}$ [mol.m$^{-3}$]")
    axs[0, 0].set_xlabel("x [µm]")
    axs[0, 0].grid(True)

    # Plot 0,1: Solid Surface Concentration
    axs[0, 1].plot(data['x_coord_cathode']*1e6, data['c_s_cathode'], color=colors[i], label=labels[i])
    axs[0, 1].plot(data['x_coord_separator']*1e6, data['c_s_separator'], color=colors[i])
    axs[0, 1].plot(data['x_coord_anode']*1e6, data['c_s_anode'], color=colors[i])
    axs[0, 1].axvspan(0, data['x_coord_separator'][0]*1e6, alpha=0.05, color='gray')
    axs[0, 1].axvspan(data['x_coord_separator'][-1]*1e6, data['x_coord_cathode'][-1]*1e6, alpha=0.05, color='orange')
    axs[0, 1].set_ylabel(r"$c_{s,surf}$ [mol.m$^{-3}$]")
    axs[0, 1].set_title("Solid Surface Concentration")
    axs[0, 1].set_xlabel("x [µm]")
    axs[0, 1].grid(True)

    # Plot 1,0: Electrolyte Potential
    axs[1, 0].plot(data['x_coord']*1e6, data['phi_ely'], color=colors[i], label=labels[i])
    axs[1, 0].axvspan(0, data['x_coord_separator'][0]*1e6, alpha=0.05, color='gray')
    axs[1, 0].axvspan(data['x_coord_separator'][-1]*1e6, data['x_coord_cathode'][-1]*1e6, alpha=0.05, color='orange')
    axs[1, 0].set_ylabel(r"$\phi_{ely}$ [V]")
    axs[1, 0].set_title(r"Electrolyte Potential")
    axs[1, 0].set_xlabel("x [µm]")
    axs[1, 0].grid(True)

    # Plot 1,1: Solid Potential
    axs[1, 1].plot(data['x_coord_cathode']*1e6, data['phi_s_cathode'], color=colors[i], label=labels[i])
    axs[1, 1].plot(data['x_coord_separator']*1e6, data['phi_s_separator'], color=colors[i])
    axs[1, 1].plot(data['x_coord_anode']*1e6, data['phi_s_anode'], color=colors[i])
    if i == 0:
        axs[1, 1].axvspan(0, data['x_coord_separator'][0]*1e6, alpha=0.05, color='gray', label='Anode')
        axs[1, 1].axvspan(data['x_coord_separator'][-1]*1e6, data['x_coord_cathode'][-1]*1e6, alpha=0.05, color='orange', label='Cathode')
    else:
        axs[1, 1].axvspan(0, data['x_coord_separator'][0]*1e6, alpha=0.05, color='gray', label='_nolegend_')
        axs[1, 1].axvspan(data['x_coord_separator'][-1]*1e6, data['x_coord_cathode'][-1]*1e6, alpha=0.05, color='orange', label='_nolegend_')
    axs[1, 1].set_ylabel(r"$\phi_s$ [V]")
    axs[1, 1].set_title(r"Solid Potential")
    axs[1, 1].set_xlabel("x [µm]")
    axs[1, 1].grid(True)

    
    

# Add legend
# handles, labels_list = axs[1, 1].get_legend_handles_labels()
# fig.legend(handles, labels_list, loc='upper center', bbox_to_anchor=(0.5, -0.02), ncol=3)
handles, labels_list = axs[1, 1].get_legend_handles_labels()
special = ["Cathode", "10 MPa"]
normal_h, normal_l, special_h, special_l = [], [], [], []
for h, l in zip(handles, labels_list):
    if l in special:
        special_h.append(h); special_l.append(l)
    else:
        normal_h.append(h); normal_l.append(l)
handles_ordered = normal_h + special_h
labels_ordered = normal_l + special_l
fig.legend(handles_ordered, labels_ordered, loc='upper center', bbox_to_anchor=(0.5, -0.02), ncol=3)

plt.tight_layout(rect=[0, 0.03, 1, 0.98])
plt.show()

SOC 0%

In [ ]:
# -----------------------
# Plotting results at last time step
# -----------------------
param_list = [param0, param1, param2]
sol_list = [sol_DFN, sol_DFN_1, sol_DFN_2]
time_frame = -1 

# Create a single figure with 2x2 subplots
fig, axs = plt.subplots(2, 2, figsize=(14, 10))
# fig.suptitle(f"Spatial Profiles Comparison - Time Step: {time_frame} (C/50)", fontsize=14)

# Colors and labels for each solution
colors = ['C0', 'C1', 'C2']
labels = ['0 Pa', '1 MPa', '10 MPa']

# Collect data from all solutions
all_data = []
for i, (param, sol) in enumerate(zip(param_list, sol_list)):
    c_ely   = sol["Electrolyte concentration [mol.m-3]"].entries[:, time_frame]
    phi_ely = sol['Electrolyte potential [V]'].data[:, time_frame]
    c_s_cathode     = sol["Positive particle surface concentration [mol.m-3]"].data[:, time_frame]
    c_s_separator     = np.zeros(var_pts['x_s'])
    c_s_anode     = sol["Negative particle surface concentration [mol.m-3]"].data[:, time_frame]
    phi_s_cathode   = sol['Positive electrode potential [V]'].data[:, time_frame]
    phi_s_separator   = np.zeros(var_pts['x_s'])
    phi_s_anode     = sol['Negative electrode potential [V]'].data[:, time_frame]

    # Build spatial coordinates
    x_coord = sol["x [m]"].entries[:, time_frame]
    x_coord_cathode = sol["x_p [m]"].entries[:, time_frame]
    x_coord_separator = sol["x_s [m]"].entries[:, time_frame]
    x_coord_anode = sol["x_n [m]"].entries[:, time_frame]

    all_data.append({
        'c_ely': c_ely,
        'phi_ely': phi_ely,
        'c_s_cathode': c_s_cathode,
        'c_s_separator': c_s_separator,
        'c_s_anode': c_s_anode,
        'phi_s_cathode': phi_s_cathode,
        'phi_s_separator': phi_s_separator,
        'phi_s_anode': phi_s_anode,
        'x_coord': x_coord,
        'x_coord_cathode': x_coord_cathode,
        'x_coord_separator': x_coord_separator,
        'x_coord_anode': x_coord_anode,
    })

# Plot all solutions on each subplot
for i, data in enumerate(all_data):
    # Plot 0,0: Electrolyte Concentration
    axs[0, 0].plot(data['x_coord']*1e6, data['c_ely'], color=colors[i], label=labels[i])
    axs[0, 0].axvspan(0, data['x_coord_separator'][0]*1e6, alpha=0.05, color='gray')
    axs[0, 0].axvspan(data['x_coord_separator'][-1]*1e6, data['x_coord_cathode'][-1]*1e6, alpha=0.05, color='orange')
    axs[0, 0].set_title("Electrolyte Concentration")
    axs[0, 0].set_ylabel(r"$c_{ely}$ [mol.m$^{-3}$]")
    axs[0, 0].set_xlabel("x [µm]")
    axs[0, 0].grid(True)

    # Plot 0,1: Solid Surface Concentration
    axs[0, 1].plot(data['x_coord_cathode']*1e6, data['c_s_cathode'], color=colors[i], label=labels[i])
    axs[0, 1].plot(data['x_coord_separator']*1e6, data['c_s_separator'], color=colors[i])
    axs[0, 1].plot(data['x_coord_anode']*1e6, data['c_s_anode'], color=colors[i])
    axs[0, 1].axvspan(0, data['x_coord_separator'][0]*1e6, alpha=0.05, color='gray')
    axs[0, 1].axvspan(data['x_coord_separator'][-1]*1e6, data['x_coord_cathode'][-1]*1e6, alpha=0.05, color='orange')
    axs[0, 1].set_ylabel(r"$c_{s,surf}$ [mol.m$^{-3}$]")
    axs[0, 1].set_title("Solid Surface Concentration")
    axs[0, 1].set_xlabel("x [µm]")
    axs[0, 1].grid(True)

    # Plot 1,0: Electrolyte Potential
    axs[1, 0].plot(data['x_coord']*1e6, data['phi_ely'], color=colors[i], label=labels[i])
    axs[1, 0].axvspan(0, data['x_coord_separator'][0]*1e6, alpha=0.05, color='gray')
    axs[1, 0].axvspan(data['x_coord_separator'][-1]*1e6, data['x_coord_cathode'][-1]*1e6, alpha=0.05, color='orange')
    axs[1, 0].set_ylabel(r"$\phi_{ely}$ [V]")
    axs[1, 0].set_title(r"Electrolyte Potential")
    axs[1, 0].set_xlabel("x [µm]")
    axs[1, 0].grid(True)

    # Plot 1,1: Solid Potential
    axs[1, 1].plot(data['x_coord_cathode']*1e6, data['phi_s_cathode'], color=colors[i], label=labels[i])
    axs[1, 1].plot(data['x_coord_separator']*1e6, data['phi_s_separator'], color=colors[i])
    axs[1, 1].plot(data['x_coord_anode']*1e6, data['phi_s_anode'], color=colors[i])
    if i == 0:
        axs[1, 1].axvspan(0, data['x_coord_separator'][0]*1e6, alpha=0.05, color='gray', label='Anode')
        axs[1, 1].axvspan(data['x_coord_separator'][-1]*1e6, data['x_coord_cathode'][-1]*1e6, alpha=0.05, color='orange', label='Cathode')
    else:
        axs[1, 1].axvspan(0, data['x_coord_separator'][0]*1e6, alpha=0.05, color='gray', label='_nolegend_')
        axs[1, 1].axvspan(data['x_coord_separator'][-1]*1e6, data['x_coord_cathode'][-1]*1e6, alpha=0.05, color='orange', label='_nolegend_')
    axs[1, 1].set_ylabel(r"$\phi_s$ [V]")
    axs[1, 1].set_title(r"Solid Potential")
    axs[1, 1].set_xlabel("x [µm]")
    axs[1, 1].grid(True)

    
    

# Add legend
# handles, labels_list = axs[1, 1].get_legend_handles_labels()
# fig.legend(handles, labels_list, loc='upper center', bbox_to_anchor=(0.5, -0.02), ncol=3)
handles, labels_list = axs[1, 1].get_legend_handles_labels()
special = ["Cathode", "10 MPa"]
normal_h, normal_l, special_h, special_l = [], [], [], []
for h, l in zip(handles, labels_list):
    if l in special:
        special_h.append(h); special_l.append(l)
    else:
        normal_h.append(h); normal_l.append(l)
handles_ordered = normal_h + special_h
labels_ordered = normal_l + special_l
fig.legend(handles_ordered, labels_ordered, loc='upper center', bbox_to_anchor=(0.5, -0.02), ncol=3)

plt.tight_layout(rect=[0, 0.03, 1, 0.98])
plt.show()

In [ ]:
# -----------------------
# Plotting results at last time step
# -----------------------
time_frame = 1

c_ely   = sol_DFN["Electrolyte concentration [mol.m-3]"].entries[:, time_frame]

phi_ely = sol_DFN['Electrolyte potential [V]'].data[:, time_frame]

c_s_cathode     = sol_DFN["Positive particle surface concentration [mol.m-3]"].data[:, time_frame]
c_s_separator     = np.zeros(var_pts['x_s'])
c_s_anode     = sol_DFN["Negative particle surface concentration [mol.m-3]"].data[:, time_frame]

phi_s_cathode   = sol_DFN['Positive electrode potential [V]'].data[:, time_frame]
phi_s_separator   = np.zeros(var_pts['x_s'])
phi_s_anode     = sol_DFN['Negative electrode potential [V]'].data[:, time_frame]


# Build spatial coordinates
x_coord = sol_DFN["x [m]"].entries[:, time_frame]
x_coord_cathode = sol_DFN["x_p [m]"].entries[:, time_frame]
x_coord_separator = sol_DFN["x_s [m]"].entries[:, time_frame]
x_coord_anode = sol_DFN["x_n [m]"].entries[:, time_frame]



# --------------------------------------------------------------------------------------
# FIGURE 1: Spatial Profiles (Ostane vizualno isto, uporabi zgornje spremenljivke)
# --------------------------------------------------------------------------------------
fig1, axs = plt.subplots(2, 2, figsize=(10, 7))
fig1.suptitle(f"Spatial Profiles at Time Step: {time_frame} (C/50)", fontsize=14)

axs[0, 0].plot(x_coord*1e6, c_ely)
axs[0, 0].axvspan(0, x_coord_separator[0]*1e6, alpha=0.1, color='gray', label="Anode")
axs[0, 0].axvspan(x_coord_separator[-1]*1e6, x_coord_cathode[-1]*1e6, alpha=0.1, color='orange', label="Cathode")
axs[0, 0].set_title("Electrolyte Concentration ($c_{ely}$)")
axs[0, 0].set_ylabel(r"$c_{ely}$ [mol.m$^{-3}$]")
axs[0, 0].legend()

axs[0, 1].plot(x_coord_cathode*1e6, c_s_cathode, color='C0')
axs[0, 1].plot(x_coord_separator*1e6, c_s_separator, color='C0')
axs[0, 1].plot(x_coord_anode*1e6, c_s_anode, color='C0')
axs[0, 1].axvspan(0, x_coord_separator[0]*1e6, alpha=0.1, color='gray', label="Anode")
axs[0, 1].axvspan(x_coord_separator[-1]*1e6, x_coord_cathode[-1]*1e6, alpha=0.1, color='orange', label="Cathode")
axs[0, 1].set_ylabel(r"$c_{s,surf}$ [mol.m$^{-3}$]")
axs[0, 1].set_title("Solid Surface Concentration ($c_{s,surf}$)")

axs[1, 0].plot(x_coord*1e6, phi_ely)
axs[1, 0].axvspan(0, x_coord_separator[0]*1e6, alpha=0.1, color='gray', label="Anode")
axs[1, 0].axvspan(x_coord_separator[-1]*1e6, x_coord_cathode[-1]*1e6, alpha=0.1, color='orange', label="Cathode")
axs[1, 0].set_ylabel(r"$\phi_{ely}$ [V]")
axs[1, 0].set_title(r"Electrolyte Potential ($\phi_{ely}$)")

axs[1, 1].plot(x_coord_cathode*1e6, phi_s_cathode, color='C0')
axs[1, 1].plot(x_coord_separator*1e6, phi_s_separator, color='C0')
axs[1, 1].plot(x_coord_anode*1e6, phi_s_anode, color='C0')
axs[1, 1].axvspan(0, x_coord_separator[0]*1e6, alpha=0.1, color='gray', label="Anode")
axs[1, 1].axvspan(x_coord_separator[-1]*1e6, x_coord_cathode[-1]*1e6, alpha=0.1, color='orange', label="Cathode")
axs[1, 1].set_ylabel(r"$\phi_s$ [V]")
axs[1, 1].set_title(r"Solid Potential ($\phi_s$)")

for ax in axs.flat:
    ax.set_xlabel("x [µm]")
    ax.grid(True)
plt.tight_layout(rect=[0, 0.03, 1, 0.98])


plt.show()

# Notranje napetosti

$$
\sigma_{rr}^c(r) = \frac{2 \Omega E_p}{3(1 - \nu_p)} \left( \frac{1}{r_p^3} \int_0^{r_p} \tilde{c} r^2 \, dr - \frac{1}{r^3} \int_0^r \tilde{c} r^2 \, dr \right),
$$

$$
\sigma_{\theta\theta}^c(r) = \frac{\Omega E_p}{3(1 - \nu_p)} \left( \frac{2}{r_p^3} \int_0^{r_p} \tilde{c} r^2 \, dr + \frac{1}{r^3} \int_0^r \tilde{c} r^2 \, dr - \tilde{c} \right),
$$

$$
\sigma_h^c(r) = \frac{\sigma_{rr}^c + 2\sigma_{\theta\theta}^c}{3} = \frac{2 \Omega E_p}{3(1 - \nu_p)} \left( \frac{1}{r_p^3} \int_0^{r_p} \tilde{c} r^2 \, dr - \frac{\tilde{c}}{3} \right)
$$

$$\tilde{c} = c_s - c_{s0}$$

$r_p$ ... radij delca \
$𝐸_𝑝$ ... Youngov modul delca \
$𝑣_𝑝$ ... Poissonov količnik delca \
$\sigma_{rr}^c$ ... radialna napetost delca \
$\sigma_{\theta\theta}^c$ ... tangencialna napetost delca \
$c_s$ ... koncentracija v trdnini \
$c_{s0}$ ... koncentracija v začetnem oz. neobremenjenem stanju 

PyBaMM:

$$
\sigma_h^c(r) = \frac{2 \Omega E_p}{3(1 - \nu_p)} \left( \frac{1}{r_p^3} \int_0^{r_p} c r^2 \, dr - \frac{c}{3} \right)
$$

$$\boxed{\tilde{c} = c}$$


## Anoda

### Omega = 3.1e-6

In [ ]:
# Formatiranje grafov
# Nastavitev velikosti grafa (širina, višina v inch-ih)
fig_size = figsize=(8, 6)  

# Nastavitev velikosti pisave
plt.rcParams.update({
    'font.size': 12,           # Osnovna velikost pisave
    'axes.titlesize': 16,      # Velikost naslova grafa
    'axes.labelsize': 14,      # Velikost oznak osi
    'xtick.labelsize': 12,     # Velikost oznak na x osi
    'ytick.labelsize': 12,     # Velikost oznak na y osi
    'legend.fontsize': 12,     # Velikost pisave v legendi
})

In [ ]:
print("0 Pa:" + str(sol_DFN["Negative particle concentration [mol.m-3]"].data.shape))
print("1MPa:" + str(sol_DFN_1["Negative particle concentration [mol.m-3]"].data.shape))
print("10MPa:" + str(sol_DFN_2["Negative particle concentration [mol.m-3]"].data.shape))


In [ ]:
print("0 Pa:" + str(sol_DFN_C["Negative particle concentration [mol.m-3]"].data.shape))
print("1MPa:" + str(sol_DFN_1_C["Negative particle concentration [mol.m-3]"].data.shape))
print("10MPa:" + str(sol_DFN_2_C["Negative particle concentration [mol.m-3]"].data.shape))
    

In [ ]:
# Seznami parametrov in rešitev za vse tri modele
param_list = [param0, param1, param2]
sol_list = [sol_DFN, sol_DFN_1, sol_DFN_2]
sigma_h_n_list = []

# Zanka čez vse modele
for param_i, sol_i in zip(param_list, sol_list):
    # Parametri modela
    E_p = 15e9  
    nu_p = 0.3  
    # Omega = param_i["Negative electrode partial molar volume [m3.mol-1]"]
    # Omega = 3.1e-4 # tako smo mi predpisali za eis
    Omega = 3.1e-6 # tako je v Mohtat2020_Mech
    c_s0 = param_i["Initial concentration in negative electrode [mol.m-3]"]
    
    # Podatki iz rešitve
    c_s = sol_i["Negative particle concentration [mol.m-3]"].data
    c_tilde = c_s - c_s0
    c_s_rav = sol_i["R-averaged negative particle concentration [mol.m-3]"].data
    
    
    # Izračun sigma_h
    sigma_h_n = 2 * Omega * E_p / (3 * (1 - nu_p)) * ((c_s_rav - c_s0)/3 - c_tilde/3)
    sigma_h_n_list.append(sigma_h_n[:, :, :])

In [ ]:
time_window_0 = 0
time_window_1 = 178000
step = 1000

In [ ]:

# --- Plot ---
r_max = param0["Negative particle radius [m]"]
x = np.linspace(0, r_max, 25)

# --- Priprava podatkov ---
model_labels = ["0 Pa", "1 MPa", "10 MPa"]
line_styles = ["-", "--", "-."] 
colors = ["C0", "C1", "C2"]
sigma_vals = [s[:, -1, time_window_0:time_window_1:step] for s in sigma_h_n_list]  # izvlečemo časovni presek pri x=L (pri CC)
time_array = sol_DFN["Time [s]"].entries[time_window_0:time_window_1:step]  # krajši časovni vektor

# --- Priprava figure ---
fig, ax = plt.subplots(figsize=(8, 5))
lines = [ax.plot([], [], label=label, linestyle=ls, color=col, linewidth=2)[0]
         for label, ls, col in zip(model_labels, line_styles, colors)]

# Nastavi meje
x_margin = 0.05 * (x[-1] - x[0])
ax.set_xlim(x[0] - x_margin, x[-1] + x_margin)

y_all = np.stack(sigma_vals)
y_min, y_max = np.min(y_all), np.max(y_all)
y_margin = 0.1 * (y_max - y_min)
ax.set_ylim(y_min - y_margin, y_max + y_margin)

ax.set_xlabel("Radial position [m]")
ax.set_ylabel(r"$\sigma_h$ [Pa]")
ax.legend()
plt.tight_layout()
plt.subplots_adjust(top=0.9)

# --- Inicializacija ---
def init():
    for line in lines:
        line.set_data([], [])
    ax.set_title("Starting...")
    return lines

# --- Posodabljanje ---
def update(frame):
    for line, sigma in zip(lines, sigma_vals):
        line.set_data(x, sigma[:, frame])
    ax.set_title(f"Negative particle $\sigma_h$ at t={time_array[frame]:.2f} s")
    return lines

# --- Animacija ---
ani = FuncAnimation(fig, update, frames=sigma_vals[0].shape[1], init_func=init, blit=True)
plt.close()

# Za prikaz v Jupyterju:
HTML(ani.to_jshtml())


In [ ]:
time_window_0 = 0
time_window_1 = 400
step = 1

In [ ]:

# --- Plot ---
r_max = param0["Negative particle radius [m]"]
x = np.linspace(0, r_max, 25)

# --- Priprava podatkov ---
model_labels = ["0 Pa", "1 MPa", "10 MPa"]
line_styles = ["-", "--", "-."] 
colors = ["C0", "C1", "C2"]
sigma_vals = [s[:, -1, time_window_0:time_window_1:step] for s in sigma_h_n_list]  # izvlečemo časovni presek pri x=L (pri CC)
time_array = sol_DFN["Time [s]"].entries[time_window_0:time_window_1:step]  # krajši časovni vektor

# --- Priprava figure ---
fig, ax = plt.subplots(figsize=(8, 5))
lines = [ax.plot([], [], label=label, linestyle=ls, color=col, linewidth=2)[0]
         for label, ls, col in zip(model_labels, line_styles, colors)]

# Nastavi meje
x_margin = 0.05 * (x[-1] - x[0])
ax.set_xlim(x[0] - x_margin, x[-1] + x_margin)

y_all = np.stack(sigma_vals)
y_min, y_max = np.min(y_all), np.max(y_all)
y_margin = 0.1 * (y_max - y_min)
ax.set_ylim(y_min - y_margin, y_max + y_margin)

ax.set_xlabel("Radial position [m]")
ax.set_ylabel(r"$\sigma_h$ [Pa]")
ax.legend()
plt.tight_layout()
plt.subplots_adjust(top=0.9)

# --- Inicializacija ---
def init():
    for line in lines:
        line.set_data([], [])
    ax.set_title("Starting...")
    return lines

# --- Posodabljanje ---
def update(frame):
    for line, sigma in zip(lines, sigma_vals):
        line.set_data(x, sigma[:, frame])
    ax.set_title(f"Negative particle $\sigma_h$ at t={time_array[frame]:.2f} s")
    return lines

# --- Animacija ---
ani = FuncAnimation(fig, update, frames=sigma_vals[0].shape[1], init_func=init, blit=True)
plt.close()

# Za prikaz v Jupyterju:
HTML(ani.to_jshtml())


In [ ]:
time_window_0 = 66500
time_window_1 = 86500
step = 100

In [ ]:

# --- Plot ---
r_max = param0["Negative particle radius [m]"]
x = np.linspace(0, r_max, 25)

# --- Priprava podatkov ---
model_labels = ["0 Pa", "1 MPa", "10 MPa"]
line_styles = ["-", "--", "-."] 
colors = ["C0", "C1", "C2"]
sigma_vals = [s[:, 0, time_window_0:time_window_1:step] for s in sigma_h_n_list]  # izvlečemo časovni presek pri x=0
time_array = sol_DFN["Time [s]"].entries[time_window_0:time_window_1:step]  # krajši časovni vektor

# --- Priprava figure ---
fig, ax = plt.subplots(figsize=(8, 5))
lines = [ax.plot([], [], label=label, linestyle=ls, color=col, linewidth=2)[0]
         for label, ls, col in zip(model_labels, line_styles, colors)]

# Nastavi meje
x_margin = 0.05 * (x[-1] - x[0])
ax.set_xlim(x[0] - x_margin, x[-1] + x_margin)

y_all = np.stack(sigma_vals)
y_min, y_max = np.min(y_all), np.max(y_all)
y_margin = 0.1 * (y_max - y_min)
ax.set_ylim(y_min - y_margin, y_max + y_margin)

ax.set_xlabel("Radial position [m]")
ax.set_ylabel(r"$\sigma_h$ [Pa]")
ax.legend()
plt.tight_layout()
plt.subplots_adjust(top=0.9)

# --- Inicializacija ---
def init():
    for line in lines:
        line.set_data([], [])
    ax.set_title("Starting...")
    return lines

# --- Posodabljanje ---
def update(frame):
    for line, sigma in zip(lines, sigma_vals):
        line.set_data(x, sigma[:, frame])
    ax.set_title(f"Negative particle $\sigma_h$ at t={time_array[frame]:.2f} s")
    return lines

# --- Animacija ---
ani = FuncAnimation(fig, update, frames=sigma_vals[0].shape[1], init_func=init, blit=True)
plt.close()

# Za prikaz v Jupyterju:
HTML(ani.to_jshtml())


Elektroda

In [ ]:
time_window_0 = 0
time_window_1 = 179000
step = 1000

In [ ]:
# --- Plot ---
x_max = param0["Negative electrode thickness [m]"]
x = np.linspace(0, x_max, var_pts['x_n'])  # 20 points for electrode thickness

# --- Priprava podatkov ---
model_labels = ["0 Pa", "1 MPa", "10 MPa"]
line_styles = ["-", "--", "-."]
colors = ["C0", "C1", "C2"]
sigma_vals = [s[-1, :, time_window_0:time_window_1:step] for s in sigma_h_n_list]  # gledamo surface napetost
time_array = sol_DFN["Time [s]"].entries[time_window_0:time_window_1:step]  # krajši časovni vektor

# --- Priprava figure ---
fig, ax = plt.subplots(figsize=(8, 5))
lines = [ax.plot([], [], label=label, linestyle=ls, color=col, linewidth=2)[0]
         for label, ls, col in zip(model_labels, line_styles, colors)]

# Nastavi meje
x_margin = 0.05 * (x[-1] - x[0])
ax.set_xlim(x[0] - x_margin, x[-1] + x_margin)

y_all = np.stack(sigma_vals)
y_min, y_max = np.min(y_all), np.max(y_all)
y_margin = 0.1 * (y_max - y_min)
ax.set_ylim(y_min - y_margin, y_max + y_margin)

ax.set_xlabel("Negative electrode position [m]")
ax.set_ylabel(r"$\sigma_h$ [Pa]")
ax.legend()
plt.tight_layout()
plt.subplots_adjust(top=0.9)

# --- Inicializacija ---
def init():
    for line in lines:
        line.set_data([], [])
    ax.set_title("Starting...")
    return lines

# --- Posodabljanje ---
def update(frame):
    for line, sigma in zip(lines, sigma_vals):
        line.set_data(x, sigma[:, frame])
    ax.set_title(f"$\sigma_h$ at t={time_array[frame]:.2f} s")
    return lines

# --- Animacija ---
ani = FuncAnimation(fig, update, frames=sigma_vals[0].shape[1], init_func=init, blit=True)
plt.close()

# Za prikaz v Jupyterju:
HTML(ani.to_jshtml())


In [ ]:
time_window_0 = 0
time_window_1 = 400
step = 1

In [ ]:
# --- Plot ---
x_max = param0["Negative electrode thickness [m]"]
x = np.linspace(0, x_max, var_pts['x_n'])  # 20 points for electrode thickness

# --- Priprava podatkov ---
model_labels = ["0 Pa", "1 MPa", "10 MPa"]
line_styles = ["-", "--", "-."]
colors = ["C0", "C1", "C2"]
sigma_vals = [s[-1, :, time_window_0:time_window_1:step] for s in sigma_h_n_list]  # gledamo surface napetost
time_array = sol_DFN["Time [s]"].entries[time_window_0:time_window_1:step]  # krajši časovni vektor

# --- Priprava figure ---
fig, ax = plt.subplots(figsize=(8, 5))
lines = [ax.plot([], [], label=label, linestyle=ls, color=col, linewidth=2)[0]
         for label, ls, col in zip(model_labels, line_styles, colors)]

# Nastavi meje
x_margin = 0.05 * (x[-1] - x[0])
ax.set_xlim(x[0] - x_margin, x[-1] + x_margin)

y_all = np.stack(sigma_vals)
y_min, y_max = np.min(y_all), np.max(y_all)
y_margin = 0.1 * (y_max - y_min)
ax.set_ylim(y_min - y_margin, y_max + y_margin)

ax.set_xlabel("Negative electrode position [m]")
ax.set_ylabel(r"$\sigma_h$ [Pa]")
ax.legend()
plt.tight_layout()
plt.subplots_adjust(top=0.9)

# --- Inicializacija ---
def init():
    for line in lines:
        line.set_data([], [])
    ax.set_title("Starting...")
    return lines

# --- Posodabljanje ---
def update(frame):
    for line, sigma in zip(lines, sigma_vals):
        line.set_data(x, sigma[:, frame])
    ax.set_title(f"$\sigma_h$ at t={time_array[frame]:.2f} s")
    return lines

# --- Animacija ---
ani = FuncAnimation(fig, update, frames=sigma_vals[0].shape[1], init_func=init, blit=True)
plt.close()

# Za prikaz v Jupyterju:
HTML(ani.to_jshtml())


In [ ]:
time_window_0 = 55500
time_window_1 = 86500
step = 100

In [ ]:
# --- Plot ---
x_max = param0["Negative electrode thickness [m]"]
x = np.linspace(0, x_max, var_pts['x_n'])  # 20 points for electrode thickness

# --- Priprava podatkov ---
model_labels = ["0 Pa", "1 MPa", "10 MPa"]
line_styles = ["-", "--", "-."]
colors = ["C0", "C1", "C2"]
sigma_vals = [s[-1, :, time_window_0:time_window_1:step] for s in sigma_h_n_list]  # gledamo surface napetost
time_array = sol_DFN["Time [s]"].entries[time_window_0:time_window_1:step]  # krajši časovni vektor

# --- Priprava figure ---
fig, ax = plt.subplots(figsize=(8, 5))
lines = [ax.plot([], [], label=label, linestyle=ls, color=col, linewidth=2)[0]
         for label, ls, col in zip(model_labels, line_styles, colors)]

# Nastavi meje
x_margin = 0.05 * (x[-1] - x[0])
ax.set_xlim(x[0] - x_margin, x[-1] + x_margin)

y_all = np.stack(sigma_vals)
y_min, y_max = np.min(y_all), np.max(y_all)
y_margin = 0.1 * (y_max - y_min)
ax.set_ylim(y_min - y_margin, y_max + y_margin)

ax.set_xlabel("Negative electrode position [m]")
ax.set_ylabel(r"$\sigma_h$ [Pa]")
ax.legend()
plt.tight_layout()
plt.subplots_adjust(top=0.9)

# --- Inicializacija ---
def init():
    for line in lines:
        line.set_data([], [])
    ax.set_title("Starting...")
    return lines

# --- Posodabljanje ---
def update(frame):
    for line, sigma in zip(lines, sigma_vals):
        line.set_data(x, sigma[:, frame])
    ax.set_title(f"$\sigma_h$ at t={time_array[frame]:.2f} s")
    return lines

# --- Animacija ---
ani = FuncAnimation(fig, update, frames=sigma_vals[0].shape[1], init_func=init, blit=True)
plt.close()

# Za prikaz v Jupyterju:
HTML(ani.to_jshtml())


Prištejemo še tlake v celici.

In [ ]:
sigma_h_n_0Pa = sigma_h_n_list[0]
sigma_h_n_1MPa = sigma_h_n_list[1] + 1e6
sigma_h_n_10MPa = sigma_h_n_list[2] + 1e7

sigma_h_n_list = [sigma_h_n_0Pa, sigma_h_n_1MPa, sigma_h_n_10MPa]

In [ ]:
time_window_0 = 0
time_window_1 = 179000
step = 1000

In [ ]:
# --- Plot ---
x_max = param0["Negative electrode thickness [m]"]
x = np.linspace(0, x_max, var_pts['x_n'])  # 20 points for electrode thickness

# --- Priprava podatkov ---
model_labels = ["0 Pa", "1 MPa", "10 MPa"]
line_styles = ["-", "--", "-."]
colors = ["C0", "C1", "C2"]
sigma_vals = [s[-1, :, time_window_0:time_window_1:step] for s in sigma_h_n_list]  # gledamo surface napetost
time_array = sol_DFN["Time [s]"].entries[time_window_0:time_window_1:step]  # krajši časovni vektor

# --- Priprava figure ---
fig, ax = plt.subplots(figsize=(8, 5))
lines = [ax.plot([], [], label=label, linestyle=ls, color=col, linewidth=2)[0]
         for label, ls, col in zip(model_labels, line_styles, colors)]

# Nastavi meje
x_margin = 0.05 * (x[-1] - x[0])
ax.set_xlim(x[0] - x_margin, x[-1] + x_margin)

y_all = np.stack(sigma_vals)
y_min, y_max = np.min(y_all), np.max(y_all)
y_margin = 0.1 * (y_max - y_min)
ax.set_ylim(y_min - y_margin, y_max + y_margin)

ax.set_xlabel("Negative electrode position [m]")
ax.set_ylabel(r"$\sigma_h$ [Pa]")
ax.legend()
plt.tight_layout()
plt.subplots_adjust(top=0.9)

# --- Inicializacija ---
def init():
    for line in lines:
        line.set_data([], [])
    ax.set_title("Starting...")
    return lines

# --- Posodabljanje ---
def update(frame):
    for line, sigma in zip(lines, sigma_vals):
        line.set_data(x, sigma[:, frame])
    ax.set_title(f"$\sigma_h$ at t={time_array[frame]:.2f} s")
    return lines

# --- Animacija ---
ani = FuncAnimation(fig, update, frames=sigma_vals[0].shape[1], init_func=init, blit=True)
plt.close()

# Za prikaz v Jupyterju:
HTML(ani.to_jshtml())


### Omega = 3.1e-4

In [ ]:
# Seznami parametrov in rešitev za vse tri modele
param_list = [param0, param1, param2]
sol_list = [sol_DFN, sol_DFN_1, sol_DFN_2]
sigma_h_n_list = []

# Zanka čez vse modele
for param_i, sol_i in zip(param_list, sol_list):
    # Parametri modela
    E_p = 15e9  
    nu_p = 0.3  
    # Omega = param_i["Negative electrode partial molar volume [m3.mol-1]"]
    Omega = 3.1e-4 # tako smo mi predpisali za eis
    # Omega = 3.1e-6 # tako je v Mohtat2020_Mech
    c_s0 = param_i["Initial concentration in negative electrode [mol.m-3]"]
    
    # Podatki iz rešitve
    c_s = sol_i["Negative particle concentration [mol.m-3]"].data
    c_tilde = c_s - c_s0
    c_s_rav = sol_i["R-averaged negative particle concentration [mol.m-3]"].data
    
    
    # Izračun sigma_h
    sigma_h_n = 2 * Omega * E_p / (3 * (1 - nu_p)) * ((c_s_rav - c_s0)/3 - c_tilde/3)
    sigma_h_n_list.append(sigma_h_n[:, :, :])

In [ ]:
sigma_h_n_list[0].shape

In [ ]:
time_window_0 = 0
time_window_1 = 400

# --- Plot ---
r_max = param0["Negative particle radius [m]"]
x = np.linspace(0, r_max, 25)

# --- Priprava podatkov ---
model_labels = ["0 Pa", "1 MPa", "10 MPa"]
line_styles = ["-", "--", "-."] 
colors = ["C0", "C1", "C2"]
sigma_vals = [s[:, 0, time_window_0:time_window_1] for s in sigma_h_n_list]  # izvlečemo časovni presek pri x=0
time_array = sol_DFN["Time [s]"].entries[time_window_0:time_window_1]  # krajši časovni vektor

# --- Priprava figure ---
fig, ax = plt.subplots(figsize=(8, 5))
lines = [ax.plot([], [], label=label, linestyle=ls, color=col, linewidth=2)[0]
         for label, ls, col in zip(model_labels, line_styles, colors)]

# Nastavi meje
x_margin = 0.05 * (x[-1] - x[0])
ax.set_xlim(x[0] - x_margin, x[-1] + x_margin)

y_all = np.stack(sigma_vals)
y_min, y_max = np.min(y_all), np.max(y_all)
y_margin = 0.1 * (y_max - y_min)
ax.set_ylim(y_min - y_margin, y_max + y_margin)

ax.set_xlabel("Radial position [m]")
ax.set_ylabel(r"$\sigma_h$ [Pa]")
ax.legend()
plt.tight_layout()
plt.subplots_adjust(top=0.9)

# --- Inicializacija ---
def init():
    for line in lines:
        line.set_data([], [])
    ax.set_title("Starting...")
    return lines

# --- Posodabljanje ---
def update(frame):
    for line, sigma in zip(lines, sigma_vals):
        line.set_data(x, sigma[:, frame])
    ax.set_title(f"Negative particle $\sigma_h$ at t={time_array[frame]:.2f} s")
    return lines

# --- Animacija ---
ani = FuncAnimation(fig, update, frames=sigma_vals[0].shape[1], init_func=init, blit=True)
plt.close()

# Za prikaz v Jupyterju:
HTML(ani.to_jshtml())


In [ ]:
# --- Plot ---
x_max = param0["Negative electrode thickness [m]"]
x = np.linspace(0, x_max, var_pts['x_n'])  # 20 points for electrode thickness

# --- Priprava podatkov ---
model_labels = ["0 Pa", "1 MPa", "10 MPa"]
line_styles = ["-", "--", "-."]
colors = ["C0", "C1", "C2"]
sigma_vals = [s[-1, :, time_window_0:time_window_1] for s in sigma_h_n_list]  # gledamo surface napetost
time_array = sol_DFN["Time [s]"].entries[time_window_0:time_window_1]  # krajši časovni vektor

# --- Priprava figure ---
fig, ax = plt.subplots(figsize=(8, 5))
lines = [ax.plot([], [], label=label, linestyle=ls, color=col, linewidth=2)[0]
         for label, ls, col in zip(model_labels, line_styles, colors)]

# Nastavi meje
x_margin = 0.05 * (x[-1] - x[0])
ax.set_xlim(x[0] - x_margin, x[-1] + x_margin)

y_all = np.stack(sigma_vals)
y_min, y_max = np.min(y_all), np.max(y_all)
y_margin = 0.1 * (y_max - y_min)
ax.set_ylim(y_min - y_margin, y_max + y_margin)

ax.set_xlabel("Negative electrode position [m]")
ax.set_ylabel(r"$\sigma_h$ [Pa]")
ax.legend()
plt.tight_layout()
plt.subplots_adjust(top=0.9)

# --- Inicializacija ---
def init():
    for line in lines:
        line.set_data([], [])
    ax.set_title("Starting...")
    return lines

# --- Posodabljanje ---
def update(frame):
    for line, sigma in zip(lines, sigma_vals):
        line.set_data(x, sigma[:, frame])
    ax.set_title(f"$\sigma_h$ at t={time_array[frame]:.2f} s")
    return lines

# --- Animacija ---
ani = FuncAnimation(fig, update, frames=sigma_vals[0].shape[1], init_func=init, blit=True)
plt.close()

# Za prikaz v Jupyterju:
HTML(ani.to_jshtml())


In [ ]:
sigma_h_n_0Pa = sigma_h_n_list[0]
sigma_h_n_1MPa = sigma_h_n_list[1] + 1e6
sigma_h_n_10MPa = sigma_h_n_list[2] + 1e7

sigma_h_n_list = [sigma_h_n_0Pa, sigma_h_n_1MPa, sigma_h_n_10MPa]

In [ ]:
# --- Plot ---
x_max = param0["Negative electrode thickness [m]"]
x = np.linspace(0, x_max, var_pts['x_n'])  # 20 points for electrode thickness

# --- Priprava podatkov ---
model_labels = ["0 Pa", "1 MPa", "10 MPa"]
line_styles = ["-", "--", "-."]
colors = ["C0", "C1", "C2"]
sigma_vals = [s[-1, :, time_window_0:time_window_1] for s in sigma_h_n_list]  # gledamo surface napetost
time_array = sol_DFN["Time [s]"].entries[time_window_0:time_window_1]  # krajši časovni vektor

# --- Priprava figure ---
fig, ax = plt.subplots(figsize=(8, 5))
lines = [ax.plot([], [], label=label, linestyle=ls, color=col, linewidth=2)[0]
         for label, ls, col in zip(model_labels, line_styles, colors)]

# Nastavi meje
x_margin = 0.05 * (x[-1] - x[0])
ax.set_xlim(x[0] - x_margin, x[-1] + x_margin)

y_all = np.stack(sigma_vals)
y_min, y_max = np.min(y_all), np.max(y_all)
y_margin = 0.1 * (y_max - y_min)
ax.set_ylim(y_min - y_margin, y_max + y_margin)

ax.set_xlabel("Negative electrode position [m]")
ax.set_ylabel(r"$\sigma_h$ [Pa]")
ax.legend()
plt.tight_layout()
plt.subplots_adjust(top=0.9)

# --- Inicializacija ---
def init():
    for line in lines:
        line.set_data([], [])
    ax.set_title("Starting...")
    return lines

# --- Posodabljanje ---
def update(frame):
    for line, sigma in zip(lines, sigma_vals):
        line.set_data(x, sigma[:, frame])
    ax.set_title(f"$\sigma_h$ at t={time_array[frame]:.2f} s")
    return lines

# --- Animacija ---
ani = FuncAnimation(fig, update, frames=sigma_vals[0].shape[1], init_func=init, blit=True)
plt.close()

# Za prikaz v Jupyterju:
HTML(ani.to_jshtml())


## Katoda

In [ ]:
# Seznami parametrov in rešitev za vse tri modele
param_list = [param0, param1, param2]
sol_list = [sol_DFN, sol_DFN_1, sol_DFN_2]
sigma_h_p_list = []

# Zanka čez vse modele
for param_i, sol_i in zip(param_list, sol_list):
    # Parametri modela
    E_p = 199e9  
    nu_p = 0.3  
    # Omega = param_i["Negative electrode partial molar volume [m3.mol-1]"]
    Omega = 1.25e-5
    c_s0 = param_i["Initial concentration in positive electrode [mol.m-3]"]
    
    # Podatki iz rešitve
    c_s = sol_i["Positive particle concentration [mol.m-3]"].data
    c_tilde = c_s - c_s0
    c_s_rav = sol_i["R-averaged positive particle concentration [mol.m-3]"].data
    
    
    # Izračun sigma_h
    sigma_h_p = 2 * Omega * E_p / (3 * (1 - nu_p)) * ((c_s_rav - c_s0)/3 - c_tilde/3)
    sigma_h_p_list.append(sigma_h_p[:, :, :])

In [ ]:
time_window_0 = 0
time_window_1 = 400
step = 1

In [ ]:
# --- Plot ---
r_max = param0["Positive particle radius [m]"]
x = np.linspace(0, r_max, 25)

# --- Priprava podatkov ---
model_labels = ["0 Pa", "1 MPa", "10 MPa"]
line_styles = ["-", "--", "-."] 
colors = ["C0", "C1", "C2"]
sigma_vals = [s[:, 0, time_window_0:time_window_1:step] for s in sigma_h_p_list]  # izvlečemo časovni presek pri x=0
time_array = sol_DFN["Time [s]"].entries[time_window_0:time_window_1:step]  # krajši časovni vektor

# --- Priprava figure ---
fig, ax = plt.subplots(figsize=(8, 5))
lines = [ax.plot([], [], label=label, linestyle=ls, color=col, linewidth=2)[0]
         for label, ls, col in zip(model_labels, line_styles, colors)]

# Nastavi meje
x_margin = 0.05 * (x[-1] - x[0])
ax.set_xlim(x[0] - x_margin, x[-1] + x_margin)

y_all = np.stack(sigma_vals)
y_min, y_max = np.min(y_all), np.max(y_all)
y_margin = 0.1 * (y_max - y_min)
ax.set_ylim(y_min - y_margin, y_max + y_margin)

ax.set_xlabel("Radial position [m]")
ax.set_ylabel(r"$\sigma_h$ [Pa]")
ax.legend()
plt.tight_layout()
plt.subplots_adjust(top=0.9)

# --- Inicializacija ---
def init():
    for line in lines:
        line.set_data([], [])
    ax.set_title("Starting...")
    return lines

# --- Posodabljanje ---
def update(frame):
    for line, sigma in zip(lines, sigma_vals):
        line.set_data(x, sigma[:, frame])
    ax.set_title(f"Positive particle $\sigma_h$ at t={time_array[frame]:.2f} s")
    return lines

# --- Animacija ---
ani = FuncAnimation(fig, update, frames=sigma_vals[0].shape[1], init_func=init, blit=True)
plt.close()

# Za prikaz v Jupyterju:
HTML(ani.to_jshtml())


In [ ]:
# --- Plot ---
x_max = param0["Positive electrode thickness [m]"]
x = np.linspace(0, x_max, var_pts['x_p'])  # 20 points for electrode thickness

# --- Priprava podatkov ---
model_labels = ["0 Pa", "1 MPa", "10 MPa"]
line_styles = ["-", "--", "-."]
colors = ["C0", "C1", "C2"]
sigma_vals = [s[-1, :, time_window_0:time_window_1] for s in sigma_h_p_list]  # gledamo surface napetost
time_array = sol_DFN["Time [s]"].entries[time_window_0:time_window_1]  # krajši časovni vektor

# --- Priprava figure ---
fig, ax = plt.subplots(figsize=(8, 5))
lines = [ax.plot([], [], label=label, linestyle=ls, color=col, linewidth=2)[0]
         for label, ls, col in zip(model_labels, line_styles, colors)]

# Nastavi meje
x_margin = 0.05 * (x[-1] - x[0])
ax.set_xlim(x[0] - x_margin, x[-1] + x_margin)

y_all = np.stack(sigma_vals)
y_min, y_max = np.min(y_all), np.max(y_all)
y_margin = 0.1 * (y_max - y_min)
ax.set_ylim(y_min - y_margin, y_max + y_margin)

ax.set_xlabel("Positive electrode position [m]")
ax.set_ylabel(r"$\sigma_h$ [Pa]")
ax.legend()
plt.tight_layout()
plt.subplots_adjust(top=0.9)

# --- Inicializacija ---
def init():
    for line in lines:
        line.set_data([], [])
    ax.set_title("Starting...")
    return lines

# --- Posodabljanje ---
def update(frame):
    for line, sigma in zip(lines, sigma_vals):
        line.set_data(x, sigma[:, frame])
    ax.set_title(f"$\sigma_h$ at t={time_array[frame]:.2f} s")
    return lines

# --- Animacija ---
ani = FuncAnimation(fig, update, frames=sigma_vals[0].shape[1], init_func=init, blit=True)
plt.close()

# Za prikaz v Jupyterju:
HTML(ani.to_jshtml())


In [ ]:
sigma_h_p_0Pa = sigma_h_p_list[0]
sigma_h_p_1MPa = sigma_h_p_list[1] + 1e6
sigma_h_p_10MPa = sigma_h_p_list[2] + 1e7

sigma_h_p_list = [sigma_h_p_0Pa, sigma_h_p_1MPa, sigma_h_p_10MPa]

In [ ]:
# --- Plot ---
x_max = param0["Positive electrode thickness [m]"]
x = np.linspace(0, x_max, var_pts['x_p'])  # 20 points for electrode thickness

# --- Priprava podatkov ---
model_labels = ["0 Pa", "1 MPa", "10 MPa"]
line_styles = ["-", "--", "-."]
colors = ["C0", "C1", "C2"]
sigma_vals = [s[-1, :, time_window_0:time_window_1] for s in sigma_h_p_list]  # gledamo surface napetost
time_array = sol_DFN["Time [s]"].entries[time_window_0:time_window_1]  # krajši časovni vektor

# --- Priprava figure ---
fig, ax = plt.subplots(figsize=(8, 5))
lines = [ax.plot([], [], label=label, linestyle=ls, color=col, linewidth=2)[0]
         for label, ls, col in zip(model_labels, line_styles, colors)]

# Nastavi meje
x_margin = 0.05 * (x[-1] - x[0])
ax.set_xlim(x[0] - x_margin, x[-1] + x_margin)

y_all = np.stack(sigma_vals)
y_min, y_max = np.min(y_all), np.max(y_all)
y_margin = 0.1 * (y_max - y_min)
ax.set_ylim(y_min - y_margin, y_max + y_margin)

ax.set_xlabel("Positive electrode position [m]")
ax.set_ylabel(r"$\sigma_h$ [Pa]")
ax.legend()
plt.tight_layout()
plt.subplots_adjust(top=0.9)

# --- Inicializacija ---
def init():
    for line in lines:
        line.set_data([], [])
    ax.set_title("Starting...")
    return lines

# --- Posodabljanje ---
def update(frame):
    for line, sigma in zip(lines, sigma_vals):
        line.set_data(x, sigma[:, frame])
    ax.set_title(f"$\sigma_h$ at t={time_array[frame]:.2f} s")
    return lines

# --- Animacija ---
ani = FuncAnimation(fig, update, frames=sigma_vals[0].shape[1], init_func=init, blit=True)
plt.close()

# Za prikaz v Jupyterju:
HTML(ani.to_jshtml())
